In [1]:
import pandas as pd

In [2]:
data = pd.read_excel("ecommerce_business_monitoring_dataset.xlsx")

In [3]:
data.head()

,Order_ID,Date,Product_Name,Category,Region,Quantity,Sales,Profit
0,ORD100000,2024-01-01,Yoga Mat,Sports,North,2,1308.18,267.56
1,ORD100018,2024-01-01,Kids T-Shirt Pack,Clothing,North,4,2159.49,534.95
2,ORD100017,2024-01-01,"Smart LED TV 43""",Electronics,North,1,21984.14,1677.59
3,ORD100016,2024-01-01,Competitive Exam Guide,Books,West,2,692.01,92.17
4,ORD100014,2024-01-01,Skincare Combo Kit,Beauty,Central,2,2723.47,690.25


In [4]:
data.tail()

,Order_ID,Date,Product_Name,Category,Region,Quantity,Sales,Profit
13459,ORD113458,2025-12-31,Winter Jacket,Clothing,West,2,7060.93,1157.53
13460,ORD113459,2025-12-31,Competitive Exam Guide,Books,East,1,417.80,74.85
13461,ORD113460,2025-12-31,"Smart LED TV 43""",Electronics,West,1,26780.81,3540.25
13462,ORD113461,2025-12-31,Skincare Combo Kit,Beauty,North,2,3299.42,1133.73
13463,ORD113463,2025-12-31,Bluetooth Speaker,Electronics,East,1,1722.64,120.60


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Order_ID      13464 non-null  object        
 1   Date          13464 non-null  datetime64[ns]
 2   Product_Name  13464 non-null  object        
 3   Category      13464 non-null  object        
 4   Region        13464 non-null  object        
 5   Quantity      13464 non-null  int64         
 6   Sales         13464 non-null  float64       
 7   Profit        13464 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 841.6+ KB


In [6]:
data.isnull().sum()

Order_ID        0
Date            0
Product_Name    0
Category        0
Region          0
Quantity        0
Sales           0
Profit          0
dtype: int64

In [7]:
data["Date"].min(), data["Date"].max() 

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [8]:
daily_data = data.groupby("Date").agg({
    "Quantity": "sum",
    "Sales": "sum",
    "Profit": "sum",
    "Order_ID": "count"
}).reset_index()

daily_data.rename(columns={"Order_ID": "Orders"}, inplace=True)

In [9]:
daily_data.head()

,Date,Quantity,Sales,Profit,Orders
0,2024-01-01,43,70986.96,11960.31,19
1,2024-01-02,56,120041.91,14100.69,26
2,2024-01-03,52,109789.35,14042.34,23
3,2024-01-04,26,74701.54,10394.48,16
4,2024-01-05,31,55600.04,15466.74,14


In [10]:
daily_data.tail()

,Date,Quantity,Sales,Profit,Orders
726,2025-12-27,47,73702.17,17871.00,24
727,2025-12-28,51,84709.72,21021.24,29
728,2025-12-29,12,17076.16,4562.03,8
729,2025-12-30,25,44734.71,6622.04,14
730,2025-12-31,14,99781.09,12151.54,10


In [11]:
daily_data["Sales_7day_avg"] = (
    daily_data["Sales"]
    .rolling(window=7)
    .mean()
    .shift(1)
)

In [12]:
daily_data[["Date", "Sales", "Sales_7day_avg"]].head(15)

,Date,Sales,Sales_7day_avg
0,2024-01-01,70986.96,NaN
1,2024-01-02,120041.91,NaN
2,2024-01-03,109789.35,NaN
3,2024-01-04,74701.54,NaN
4,2024-01-05,55600.04,NaN
5,2024-01-06,58304.26,NaN
6,2024-01-07,54118.00,NaN
7,2024-01-08,62523.89,77648.865714
8,2024-01-09,31161.94,76439.855714
9,2024-01-10,61322.38,63742.717143


In [13]:
daily_data["Sales_change_pct"] = (
    (daily_data["Sales"] - daily_data["Sales_7day_avg"])
    / daily_data["Sales_7day_avg"]
) * 100

In [14]:
daily_data[["Date", "Sales", "Sales_7day_avg", "Sales_change_pct"]].head(15)

,Date,Sales,Sales_7day_avg,Sales_change_pct
0,2024-01-01,70986.96,NaN,NaN
1,2024-01-02,120041.91,NaN,NaN
2,2024-01-03,109789.35,NaN,NaN
3,2024-01-04,74701.54,NaN,NaN
4,2024-01-05,55600.04,NaN,NaN
5,2024-01-06,58304.26,NaN,NaN
6,2024-01-07,54118.00,NaN,NaN
7,2024-01-08,62523.89,77648.865714,-19.478682
8,2024-01-09,31161.94,76439.855714,-59.233387
9,2024-01-10,61322.38,63742.717143,-3.797041


In [15]:
daily_data["Sales_anomaly"] = daily_data["Sales_change_pct"].apply(
    lambda x: "Drop" if x < -15
    else "Spike" if x > 15
    else "Normal"
)

In [16]:
daily_data["Sales_anomaly"].value_counts()

Sales_anomaly
Drop      301
Spike     226
Normal    204
Name: count, dtype: int64

In [17]:
daily_data["Sales_change_pct"].describe()

count    724.000000
mean       2.918035
std       50.643794
min      -88.565640
25%      -30.774025
50%       -5.900613
75%       27.032742
max      426.303303
Name: Sales_change_pct, dtype: float64

In [18]:
daily_data[
    ["Date", "Sales", "Sales_7day_avg", "Sales_change_pct"]
].sort_values(
    "Sales_change_pct",
    ascending=False
).head(10)

,Date,Sales,Sales_7day_avg,Sales_change_pct
304,2024-10-31,334961.70,63644.232857,426.303303
339,2024-12-05,170547.34,45301.202857,276.474198
87,2024-03-28,134654.89,42262.240000,218.617494
47,2024-02-17,93099.23,29911.670000,211.247182
461,2025-04-06,160036.15,51890.604286,208.410650
276,2024-10-03,131692.68,45776.207143,187.688055
250,2024-09-07,154550.47,55954.422857,176.207781
523,2025-06-07,169065.50,65265.095714,159.044284
103,2024-04-13,99719.31,39456.667143,152.731204
386,2025-01-21,147770.61,59282.542857,149.264965


In [19]:
daily_data[
    ["Date", "Sales", "Sales_7day_avg", "Sales_change_pct"]
].sort_values(
    "Sales_change_pct",
    ascending=True
).head(10)

,Date,Sales,Sales_7day_avg,Sales_change_pct
260,2024-09-17,6835.19,59777.632857,-88.565640
728,2025-12-29,17076.16,100545.947143,-83.016561
414,2025-02-18,10539.11,58741.050000,-82.058356
15,2024-01-16,18044.13,83286.932857,-78.334981
514,2025-05-29,19119.72,83296.160000,-77.046097
50,2024-02-20,9906.33,41708.554286,-76.248685
484,2025-04-29,20415.26,81111.717143,-74.830689
504,2025-05-19,19196.58,70741.667143,-72.863829
301,2024-10-28,21791.81,78283.314286,-72.162893
707,2025-12-08,31830.77,108688.674286,-70.713812


In [20]:
daily_data["Sales_change_pct"].quantile([0.25, 0.75])

0.25   -30.774025
0.75    27.032742
Name: Sales_change_pct, dtype: float64

In [21]:
iqr = 27.032742 - (-30.774025)
print(iqr)

57.806767


In [22]:
#lower boundary 
lower_bound = -30.774025 - (1.5 * 57.806767)
print(lower_bound)

-117.48417549999999


In [23]:
# So our IQR method says:

#If Sales_change_pct goes below -117.48%, it's statistically unusual enough to be considered a potential anomaly.

In [24]:
#Upper boundary
upper_bound = 27.032742 + (1.5 * 57.806767)
print(upper_bound)

113.7428925


In [25]:
daily_data["Sales_anomaly"] = daily_data["Sales_change_pct"].apply(
    lambda x: "Drop" if x < lower_bound
    else "Spike" if x > upper_bound
    else "Normal"
)

In [26]:
daily_data["Sales_anomaly"].value_counts()

Sales_anomaly
Normal    709
Spike      22
Name: count, dtype: int64

In [27]:
daily_data[daily_data["Sales_anomaly"] == "Spike"][
    ["Date", "Sales", "Sales_7day_avg", "Sales_change_pct"]
]

,Date,Sales,Sales_7day_avg,Sales_change_pct
10,2024-01-11,123028.88,56818.864286,116.528228
47,2024-02-17,93099.23,29911.670000,211.247182
61,2024-03-02,109029.48,47428.654286,129.881032
87,2024-03-28,134654.89,42262.240000,218.617494
103,2024-04-13,99719.31,39456.667143,152.731204
166,2024-06-15,128959.86,51857.567143,148.680891
250,2024-09-07,154550.47,55954.422857,176.207781
276,2024-10-03,131692.68,45776.207143,187.688055
304,2024-10-31,334961.70,63644.232857,426.303303
321,2024-11-17,143362.13,59674.570000,140.239905


In [28]:
ground_truth = pd.read_excel(
    "ecommerce_business_monitoring_dataset.xlsx",
    sheet_name="Ground_Truth"
)

ground_truth

,Anomaly_ID,Start_Date,End_Date,Anomaly_Type,Affected_Product,Affected_Category,Affected_Region,Expected_Impact
0,A001,2024-02-10,2024-02-12,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 28%...
1,A002,2024-03-05,2024-03-08,Regional Sales Drop,All,All,West,Sales and Quantity decreased approximately 32%...
2,A003,2024-04-18,2024-04-20,Product Demand Decline,Wireless Earbuds Pro,Electronics,All,Sales and Quantity decreased approximately 42%...
3,A004,2024-05-22,2024-05-25,Profit Margin Drop,All,Electronics,All,Sales stayed within the normal range; Profit d...
4,A005,2024-07-14,2024-07-15,Unusual Sales Spike,All,Sports,All,Sales and order volume increased approximately...
5,A006,2024-08-27,2024-08-30,"Margin Problem (Sales Up, Profit Down)",All,Clothing,All,Sales increased approximately 15% for the affe...
6,A007,2024-10-05,2024-10-07,Regional Sales Drop,All,All,South,Sales and Quantity decreased approximately 29%...
7,A008,2024-11-11,2024-11-12,Product Sales Spike,Air Fryer,Home & Kitchen,All,Sales and order volume increased approximately...
8,A009,2025-01-09,2025-01-13,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 24%...
9,A010,2025-03-03,2025-03-05,Profit Margin Drop,All,Home & Kitchen,All,Sales stayed within the normal range; Profit d...


In [29]:
daily_data["Day"] = daily_data["Date"].dt.day_name()

In [30]:
daily_data[["Date", "Day"]].head(10)

,Date,Day
0,2024-01-01,Monday
1,2024-01-02,Tuesday
2,2024-01-03,Wednesday
3,2024-01-04,Thursday
4,2024-01-05,Friday
5,2024-01-06,Saturday
6,2024-01-07,Sunday
7,2024-01-08,Monday
8,2024-01-09,Tuesday
9,2024-01-10,Wednesday


In [31]:
daily_data.groupby("Day")

In [32]:
daily_data.groupby("Day")["Sales"].mean()


Day
Friday       65316.477308
Monday       59258.508952
Saturday     81854.570577
Sunday       75057.326442
Thursday     68045.437885
Tuesday      58251.002286
Wednesday    61011.235048
Name: Sales, dtype: float64

In [33]:
daily_data["Same_Day_7day_Avg"] = (
    daily_data.groupby("Day")["Sales"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [34]:
daily_data[["Date", "Day", "Sales", "Same_Day_7day_Avg"]].tail(20)

,Date,Day,Sales,Same_Day_7day_Avg
711,2025-12-12,Friday,44470.88,72454.0925
712,2025-12-13,Saturday,65388.41,104877.1600
713,2025-12-14,Sunday,36458.76,94314.2250
714,2025-12-15,Monday,55277.69,89459.1675
715,2025-12-16,Tuesday,78711.98,84022.3850
716,2025-12-17,Wednesday,91436.98,91345.5400
717,2025-12-18,Thursday,41300.83,75315.4375
718,2025-12-19,Friday,63655.43,56415.2225
719,2025-12-20,Saturday,135340.52,98615.3475
720,2025-12-21,Sunday,94105.34,84787.6375


In [35]:
daily_data.rename(
    columns={"Same_Day_7day_Avg": "Same_Day_4week_Avg"},
    inplace=True
)

In [36]:
daily_data["Same_Day_Change_pct"] = (
    (daily_data["Sales"] - daily_data["Same_Day_4week_Avg"])
    / daily_data["Same_Day_4week_Avg"]
) * 100

In [37]:
daily_data[
    ["Date", "Day", "Sales", "Same_Day_4week_Avg", "Same_Day_Change_pct"]
].tail(20)

,Date,Day,Sales,Same_Day_4week_Avg,Same_Day_Change_pct
711,2025-12-12,Friday,44470.88,72454.0925,-38.621990
712,2025-12-13,Saturday,65388.41,104877.1600,-37.652383
713,2025-12-14,Sunday,36458.76,94314.2250,-61.343307
714,2025-12-15,Monday,55277.69,89459.1675,-38.209027
715,2025-12-16,Tuesday,78711.98,84022.3850,-6.320226
716,2025-12-17,Wednesday,91436.98,91345.5400,0.100103
717,2025-12-18,Thursday,41300.83,75315.4375,-45.162863
718,2025-12-19,Friday,63655.43,56415.2225,12.833783
719,2025-12-20,Saturday,135340.52,98615.3475,37.240829
720,2025-12-21,Sunday,94105.34,84787.6375,10.989459


In [38]:
daily_data["Same_Day_Change_pct"].describe()

count    703.000000
mean       5.340653
std       51.199066
min      -86.602251
25%      -29.562675
50%       -4.657650
75%       31.020642
max      279.613555
Name: Same_Day_Change_pct, dtype: float64

In [39]:
Q1 = daily_data["Same_Day_Change_pct"].quantile(0.25)
Q3 = daily_data["Same_Day_Change_pct"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower:", lower_bound)
print("Upper:", upper_bound)

Q1: -29.562674863922453
Q3: 31.020642204248865
IQR: 60.58331706817132
Lower: -120.43765046617943
Upper: 121.89561780650584


In [40]:
daily_data["Same_Day_Change_pct"].std()

51.19906569721809

In [41]:
daily_data["Z_score"] = (
    (daily_data["Same_Day_Change_pct"] - daily_data["Same_Day_Change_pct"].mean())
    / daily_data["Same_Day_Change_pct"].std()
)


#Formula:

# Z= Standard Deviation Value − Mean
	


In [42]:
daily_data[[
    "Date",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct",
    "Z_score"
]].tail(10)

,Date,Sales,Same_Day_4week_Avg,Same_Day_Change_pct,Z_score
721,2025-12-22,154125.88,79975.3450,92.716743,1.706595
722,2025-12-23,150082.52,83619.5825,79.482503,1.448109
723,2025-12-24,48731.84,98814.4850,-50.683506,-1.094242
724,2025-12-25,116430.26,70647.8700,64.803638,1.161408
725,2025-12-26,76039.24,59841.2850,27.068194,0.424374
726,2025-12-27,73702.17,98186.9700,-24.936914,-0.591370
727,2025-12-28,84709.72,84263.1800,0.529935,-0.093961
728,2025-12-29,17076.16,91249.0800,-81.286211,-1.691962
729,2025-12-30,44734.71,97340.7825,-54.043199,-1.159862
730,2025-12-31,99781.09,88607.7375,12.609906,0.141980


In [43]:
daily_data["Same_Day_Change_pct"].mean()


np.float64(5.340653378251109)

In [44]:
daily_data["Same_Day_Change_pct"].std()

51.19906569721809

In [45]:
daily_data["Rolling_Mean"] = (
    daily_data["Same_Day_Change_pct"]
    .rolling(window=30)
    .mean()
    .shift(1)
)

In [46]:
daily_data[[
    "Date",
    "Same_Day_Change_pct",
    "Rolling_Mean"
]].tail(10)

,Date,Same_Day_Change_pct,Rolling_Mean
721,2025-12-22,92.716743,1.894404
722,2025-12-23,79.482503,3.449864
723,2025-12-24,-50.683506,5.431029
724,2025-12-25,64.803638,3.438597
725,2025-12-26,27.068194,4.761539
726,2025-12-27,-24.936914,4.292942
727,2025-12-28,0.529935,2.235670
728,2025-12-29,-81.286211,3.911598
729,2025-12-30,-54.043199,1.150609
730,2025-12-31,12.609906,-0.125656


In [47]:
daily_data["Rolling_Std"] = (
    daily_data["Same_Day_Change_pct"]
    .rolling(window=30)
    .std()
    .shift(1)
)

In [48]:
daily_data[[
    "Date",
    "Same_Day_Change_pct",
    "Rolling_Mean",
    "Rolling_Std"
]].tail(10)

,Date,Same_Day_Change_pct,Rolling_Mean,Rolling_Std
721,2025-12-22,92.716743,1.894404,51.446736
722,2025-12-23,79.482503,3.449864,53.492625
723,2025-12-24,-50.683506,5.431029,55.201854
724,2025-12-25,64.803638,3.438597,56.136060
725,2025-12-26,27.068194,4.761539,57.123502
726,2025-12-27,-24.936914,4.292942,56.872015
727,2025-12-28,0.529935,2.235670,56.772468
728,2025-12-29,-81.286211,3.911598,55.920707
729,2025-12-30,-54.043199,1.150609,58.046059
730,2025-12-31,12.609906,-0.125656,58.845997


In [49]:
daily_data["Rolling_Z_score"] = (
    (daily_data["Same_Day_Change_pct"] - daily_data["Rolling_Mean"])
    / daily_data["Rolling_Std"]
)

In [50]:
daily_data[[
    "Date",
    "Same_Day_Change_pct",
    "Rolling_Mean",
    "Rolling_Std",
    "Rolling_Z_score"
]].tail(10)

,Date,Same_Day_Change_pct,Rolling_Mean,Rolling_Std,Rolling_Z_score
721,2025-12-22,92.716743,1.894404,51.446736,1.765366
722,2025-12-23,79.482503,3.449864,53.492625,1.421367
723,2025-12-24,-50.683506,5.431029,55.201854,-1.016534
724,2025-12-25,64.803638,3.438597,56.136060,1.093148
725,2025-12-26,27.068194,4.761539,57.123502,0.390499
726,2025-12-27,-24.936914,4.292942,56.872015,-0.513959
727,2025-12-28,0.529935,2.235670,56.772468,-0.030045
728,2025-12-29,-81.286211,3.911598,55.920707,-1.523547
729,2025-12-30,-54.043199,1.150609,58.046059,-0.950862
730,2025-12-31,12.609906,-0.125656,58.845997,0.216422


In [51]:
daily_data["Anomaly"] = daily_data["Rolling_Z_score"].apply(
    lambda z: "Drop" if z < -2
    else "Spike" if z > 2
    else "Normal"
)

In [52]:
daily_data["Anomaly"].value_counts()

Anomaly
Normal    692
Spike      37
Drop        2
Name: count, dtype: int64

In [53]:
daily_data[daily_data["Anomaly"] != "Normal"][[
    "Date",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct",
    "Rolling_Z_score",
    "Anomaly"
]]

,Date,Sales,Same_Day_4week_Avg,Same_Day_Change_pct,Rolling_Z_score,Anomaly
61,2024-03-02,109029.48,50740.7975,114.875377,2.846166,Spike
62,2024-03-03,75052.79,31286.7500,139.886821,2.925120,Spike
66,2024-03-07,117362.94,38091.2875,208.109669,3.676347,Spike
103,2024-04-13,99719.31,43091.0400,131.415417,3.354868,Spike
106,2024-04-16,104840.26,42290.1625,147.906969,3.195657,Spike
119,2024-04-29,89046.94,33375.5775,166.802694,3.418646,Spike
123,2024-05-03,130984.56,45415.1500,188.416002,2.982935,Spike
171,2024-06-20,89103.85,51486.6350,73.062097,2.400186,Spike
176,2024-06-25,108335.55,64414.1250,68.186015,2.087403,Spike
177,2024-06-26,98331.71,32245.7850,204.944383,5.327948,Spike


In [54]:
daily_data[
    (daily_data["Date"] >= "2024-02-08") &
    (daily_data["Date"] <= "2024-02-14")
][[
    "Date",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct",
    "Rolling_Z_score",
    "Anomaly"
]]

,Date,Sales,Same_Day_4week_Avg,Same_Day_Change_pct,Rolling_Z_score,Anomaly
38,2024-02-08,56369.41,73066.0625,-22.851447,NaN,Normal
39,2024-02-09,44466.47,70784.3625,-37.180377,NaN,Normal
40,2024-02-10,27856.27,73015.3250,-61.848735,NaN,Normal
41,2024-02-11,16940.17,57132.9975,-70.349587,NaN,Normal
42,2024-02-12,32755.84,36160.8350,-9.416251,NaN,Normal
43,2024-02-13,27512.15,34992.2825,-21.376521,NaN,Normal
44,2024-02-14,45244.45,62005.4275,-27.031468,NaN,Normal


In [55]:
daily_data["Anomaly"] = daily_data["Rolling_Z_score"].apply(
    lambda z: "Insufficient_Data" if pd.isna(z)
    else "Drop" if z < -2
    else "Spike" if z > 2
    else "Normal"
)

In [56]:
daily_data["Anomaly"].value_counts()

Anomaly
Normal               634
Insufficient_Data     58
Spike                 37
Drop                   2
Name: count, dtype: int64

In [57]:
daily_data[
    (daily_data["Date"] >= "2024-03-03") &
    (daily_data["Date"] <= "2024-03-10")
][[
    "Date",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct",
    "Rolling_Z_score",
    "Anomaly"
]]

,Date,Sales,Same_Day_4week_Avg,Same_Day_Change_pct,Rolling_Z_score,Anomaly
62,2024-03-03,75052.79,31286.7500,139.886821,2.925120,Spike
63,2024-03-04,54855.93,40231.8500,36.349509,0.702657,Normal
64,2024-03-05,40523.57,34996.4200,15.793473,0.288711,Normal
65,2024-03-06,55326.05,47903.3950,15.495050,0.248535,Normal
66,2024-03-07,117362.94,38091.2875,208.109669,3.676347,Spike
67,2024-03-08,71723.74,36429.5200,96.883571,1.298505,Normal
68,2024-03-09,53116.27,71852.5725,-26.076036,-0.555119,Normal
69,2024-03-10,92068.37,41984.7825,119.289858,1.548787,Normal


In [58]:
region_daily = data.groupby(["Date", "Region"]).agg({
    "Sales": "sum",
    "Quantity": "sum",
    "Order_ID": "count"
}).reset_index()

region_daily.rename(columns={"Order_ID": "Orders"}, inplace=True)

In [59]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
]

,Date,Region,Sales,Quantity,Orders
293,2024-03-03,West,20857.92,12,5
298,2024-03-04,West,14153.44,7,4
303,2024-03-05,West,2715.28,2,1
308,2024-03-06,West,4214.41,2,2
317,2024-03-08,West,4813.72,4,2
322,2024-03-09,West,13100.11,12,4
327,2024-03-10,West,38190.81,10,6


In [60]:
all_dates = data["Date"].unique()
all_regions = data["Region"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_dates, all_regions],
    names=["Date", "Region"]
)

region_daily = (
    region_daily
    .set_index(["Date", "Region"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

In [61]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
]

,Date,Region,Sales,Quantity,Orders
311,2024-03-03,West,20857.92,12,5
316,2024-03-04,West,14153.44,7,4
321,2024-03-05,West,2715.28,2,1
326,2024-03-06,West,4214.41,2,2
331,2024-03-07,West,0.00,0,0
336,2024-03-08,West,4813.72,4,2
341,2024-03-09,West,13100.11,12,4
346,2024-03-10,West,38190.81,10,6


In [62]:
region_daily["Day"] = region_daily["Date"].dt.day_name()

region_daily["Same_Day_4week_Avg"] = (
    region_daily.groupby(["Region", "Day"])["Sales"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [63]:
region_daily["Same_Day_Change_pct"] = (
    (region_daily["Sales"] - region_daily["Same_Day_4week_Avg"])
    / region_daily["Same_Day_4week_Avg"]
) * 100

In [64]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Region",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct"
]]

,Date,Region,Sales,Same_Day_4week_Avg,Same_Day_Change_pct
311,2024-03-03,West,20857.92,6170.7625,238.012037
316,2024-03-04,West,14153.44,5845.4950,142.125603
321,2024-03-05,West,2715.28,9399.6675,-71.113021
326,2024-03-06,West,4214.41,3390.3125,24.307420
331,2024-03-07,West,0.00,9142.1375,-100.000000
336,2024-03-08,West,4813.72,9246.8875,-47.942267
341,2024-03-09,West,13100.11,14026.7625,-6.606318
346,2024-03-10,West,38190.81,9394.0500,306.542546


In [65]:
region_daily["Rolling_Mean"] = (
    region_daily.groupby("Region")["Same_Day_Change_pct"]
    .transform(lambda x: x.rolling(30).mean().shift(1))
)

region_daily["Rolling_Std"] = (
    region_daily.groupby("Region")["Same_Day_Change_pct"]
    .transform(lambda x: x.rolling(30).std().shift(1))
)

In [66]:
region_daily["Rolling_Z_score"] = (
    (region_daily["Same_Day_Change_pct"] - region_daily["Rolling_Mean"])
    / region_daily["Rolling_Std"]
)

In [67]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Region",
    "Sales",
    "Same_Day_Change_pct",
    "Rolling_Mean",
    "Rolling_Std",
    "Rolling_Z_score"
]]

,Date,Region,Sales,Same_Day_Change_pct,Rolling_Mean,Rolling_Std,Rolling_Z_score
311,2024-03-03,West,20857.92,238.012037,23.061111,126.007747,1.705855
316,2024-03-04,West,14153.44,142.125603,32.882975,130.966903,0.834124
321,2024-03-05,West,2715.28,-71.113021,37.106242,132.419363,-0.817247
326,2024-03-06,West,4214.41,24.307420,35.218268,133.578818,-0.081681
331,2024-03-07,West,0.00,-100.000000,35.348121,133.565821,-1.013344
336,2024-03-08,West,4813.72,-47.942267,32.935837,135.383636,-0.597399
341,2024-03-09,West,13100.11,-6.606318,33.756651,134.795299,-0.299439
346,2024-03-10,West,38190.81,306.542546,32.205775,134.989455,2.032283


In [68]:
region_daily["Anomaly"] = region_daily["Rolling_Z_score"].apply(
    lambda z: "Insufficient_Data" if pd.isna(z)
    else "Drop" if z < -2
    else "Spike" if z > 2
    else "Normal"
)

In [69]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Sales",
    "Same_Day_Change_pct",
    "Rolling_Z_score",
    "Anomaly"
]]

,Date,Sales,Same_Day_Change_pct,Rolling_Z_score,Anomaly
311,2024-03-03,20857.92,238.012037,1.705855,Normal
316,2024-03-04,14153.44,142.125603,0.834124,Normal
321,2024-03-05,2715.28,-71.113021,-0.817247,Normal
326,2024-03-06,4214.41,24.307420,-0.081681,Normal
331,2024-03-07,0.00,-100.000000,-1.013344,Normal
336,2024-03-08,4813.72,-47.942267,-0.597399,Normal
341,2024-03-09,13100.11,-6.606318,-0.299439,Normal
346,2024-03-10,38190.81,306.542546,2.032283,Spike


In [70]:
region_daily["Quantity_4week_Avg"] = (
    region_daily.groupby(["Region", "Day"])["Quantity"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [71]:
region_daily["Quantity_Change_pct"] = (
    (region_daily["Quantity"] - region_daily["Quantity_4week_Avg"])
    / region_daily["Quantity_4week_Avg"]
) * 100

In [72]:
region_daily["Orders_4week_Avg"] = (
    region_daily.groupby(["Region", "Day"])["Orders"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [73]:
region_daily["Orders_Change_pct"] = (
    (region_daily["Orders"] - region_daily["Orders_4week_Avg"])
    / region_daily["Orders_4week_Avg"]
) * 100

In [74]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Same_Day_Change_pct",
    "Quantity_Change_pct",
    "Orders_Change_pct"
]]

,Date,Same_Day_Change_pct,Quantity_Change_pct,Orders_Change_pct
311,2024-03-03,238.012037,182.352941,185.714286
316,2024-03-04,142.125603,115.384615,77.777778
321,2024-03-05,-71.113021,-63.636364,-71.428571
326,2024-03-06,24.307420,-38.461538,14.285714
331,2024-03-07,-100.000000,-100.000000,-100.000000
336,2024-03-08,-47.942267,-23.809524,-27.272727
341,2024-03-09,-6.606318,45.454545,-5.882353
346,2024-03-10,306.542546,90.476190,140.000000


In [75]:
def detect_regional_anomaly(row):
    signals = [
        row["Same_Day_Change_pct"] < -30,
        row["Quantity_Change_pct"] < -30,
        row["Orders_Change_pct"] < -30
    ]

    if sum(signals) >= 2:
        return "Drop"
    else:
        return "Normal"
        

In [76]:
region_daily["Multi_Signal_Anomaly"] = region_daily.apply(
    detect_regional_anomaly,
    axis=1
)

In [77]:
region_daily[
    (region_daily["Date"] >= "2024-03-03") &
    (region_daily["Date"] <= "2024-03-10") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Same_Day_Change_pct",
    "Quantity_Change_pct",
    "Orders_Change_pct",
    "Multi_Signal_Anomaly"
]]

,Date,Same_Day_Change_pct,Quantity_Change_pct,Orders_Change_pct,Multi_Signal_Anomaly
311,2024-03-03,238.012037,182.352941,185.714286,Normal
316,2024-03-04,142.125603,115.384615,77.777778,Normal
321,2024-03-05,-71.113021,-63.636364,-71.428571,Drop
326,2024-03-06,24.307420,-38.461538,14.285714,Normal
331,2024-03-07,-100.000000,-100.000000,-100.000000,Drop
336,2024-03-08,-47.942267,-23.809524,-27.272727,Normal
341,2024-03-09,-6.606318,45.454545,-5.882353,Normal
346,2024-03-10,306.542546,90.476190,140.000000,Normal


# evaluate the detector against all 14 ground-truth events

In [78]:
def detect_regional_anomaly(row):
    signals = [
        row["Same_Day_Change_pct"] < -30,
        row["Quantity_Change_pct"] < -30,
        row["Orders_Change_pct"] < -30
    ]

    if sum(signals) >= 2:
        return "Drop"
    else:
        return "Normal"

region_daily["Multi_Signal_Anomaly"] = region_daily.apply(
    detect_regional_anomaly,
    axis=1
)

In [79]:
region_daily["Multi_Signal_Anomaly"].value_counts()

Multi_Signal_Anomaly
Normal    2418
Drop      1237
Name: count, dtype: int64

In [80]:
ground_truth

,Anomaly_ID,Start_Date,End_Date,Anomaly_Type,Affected_Product,Affected_Category,Affected_Region,Expected_Impact
0,A001,2024-02-10,2024-02-12,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 28%...
1,A002,2024-03-05,2024-03-08,Regional Sales Drop,All,All,West,Sales and Quantity decreased approximately 32%...
2,A003,2024-04-18,2024-04-20,Product Demand Decline,Wireless Earbuds Pro,Electronics,All,Sales and Quantity decreased approximately 42%...
3,A004,2024-05-22,2024-05-25,Profit Margin Drop,All,Electronics,All,Sales stayed within the normal range; Profit d...
4,A005,2024-07-14,2024-07-15,Unusual Sales Spike,All,Sports,All,Sales and order volume increased approximately...
5,A006,2024-08-27,2024-08-30,"Margin Problem (Sales Up, Profit Down)",All,Clothing,All,Sales increased approximately 15% for the affe...
6,A007,2024-10-05,2024-10-07,Regional Sales Drop,All,All,South,Sales and Quantity decreased approximately 29%...
7,A008,2024-11-11,2024-11-12,Product Sales Spike,Air Fryer,Home & Kitchen,All,Sales and order volume increased approximately...
8,A009,2025-01-09,2025-01-13,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 24%...
9,A010,2025-03-03,2025-03-05,Profit Margin Drop,All,Home & Kitchen,All,Sales stayed within the normal range; Profit d...


In [81]:
region_daily[
    (region_daily["Date"] >= "2024-03-05") &
    (region_daily["Date"] <= "2024-03-08") &
    (region_daily["Region"] == "West")
][[
    "Date",
    "Same_Day_Change_pct",
    "Quantity_Change_pct",
    "Orders_Change_pct",
    "Multi_Signal_Anomaly"
]]

,Date,Same_Day_Change_pct,Quantity_Change_pct,Orders_Change_pct,Multi_Signal_Anomaly
321,2024-03-05,-71.113021,-63.636364,-71.428571,Drop
326,2024-03-06,24.307420,-38.461538,14.285714,Normal
331,2024-03-07,-100.000000,-100.000000,-100.000000,Drop
336,2024-03-08,-47.942267,-23.809524,-27.272727,Normal


In [82]:
region_daily[
    (region_daily["Date"] >= "2024-10-05") &
    (region_daily["Date"] <= "2024-10-07") &
    (region_daily["Region"] == "South")
][[
    "Date",
    "Same_Day_Change_pct",
    "Quantity_Change_pct",
    "Orders_Change_pct",
    "Multi_Signal_Anomaly"
]]

,Date,Same_Day_Change_pct,Quantity_Change_pct,Orders_Change_pct,Multi_Signal_Anomaly
1393,2024-10-05,-50.028265,-55.555556,-33.333333,Drop
1398,2024-10-06,43.356367,18.518519,33.333333,Normal
1403,2024-10-07,-20.183128,-30.434783,-20.000000,Normal


In [83]:
region_daily[
    (region_daily["Date"] >= "2025-07-07") &
    (region_daily["Date"] <= "2025-07-08") &
    (region_daily["Region"] == "East")
][[
    "Date",
    "Same_Day_Change_pct",
    "Quantity_Change_pct",
    "Orders_Change_pct",
    "Multi_Signal_Anomaly"
]]

,Date,Same_Day_Change_pct,Quantity_Change_pct,Orders_Change_pct,Multi_Signal_Anomaly
2769,2025-07-07,4.180464,12.0,-14.285714,Normal
2774,2025-07-08,55.007296,20.0,6.666667,Normal


In [84]:
region_daily[
    (region_daily["Date"] >= "2025-07-05") &
    (region_daily["Date"] <= "2025-07-10") &
    (region_daily["Region"] == "East")
][[
    "Date",
    "Sales",
    "Quantity",
    "Orders",
    "Same_Day_4week_Avg"
]]

,Date,Sales,Quantity,Orders,Same_Day_4week_Avg
2759,2025-07-05,11088.69,4,2,23886.6625
2764,2025-07-06,16038.48,13,4,13444.0325
2769,2025-07-07,9826.33,7,3,9432.0275
2774,2025-07-08,16300.42,9,4,10515.9050
2779,2025-07-09,18991.28,9,5,18227.3200
2784,2025-07-10,0.00,0,0,9888.3700


In [85]:
data[
    (data["Date"] >= "2025-07-05") &
    (data["Date"] <= "2025-07-10") &
    (data["Region"] == "East")
][[
    "Date",
    "Product_Name",
    "Quantity",
    "Sales",
    "Profit"
]].sort_values("Date")

,Date,Product_Name,Quantity,Sales,Profit
9703,2025-07-05,Non-stick Cookware Set,3,9975.63,2287.34
9704,2025-07-05,Women's Kurti,1,1113.06,369.58
9711,2025-07-06,Electric Kettle,2,2597.32,512.59
9717,2025-07-06,Yoga Mat,5,4019.15,947.40
9718,2025-07-06,Competitive Exam Guide,4,1775.99,245.95
9722,2025-07-06,Running Shoes,2,7646.02,2099.31
9738,2025-07-07,Wireless Earbuds Pro,3,7443.88,869.18
9747,2025-07-07,Fiction Bestseller Bundle,2,1189.97,194.66
9735,2025-07-07,Fiction Bestseller Bundle,2,1192.48,194.20
9752,2025-07-08,Smartphone Case & Charger Combo,5,4752.20,465.67


In [86]:
data[
    (data["Date"] >= "2025-07-01") &
    (data["Date"] <= "2025-07-14") &
    (data["Region"] == "East")
].groupby("Date").agg({
    "Quantity": "sum",
    "Sales": "sum",
    "Profit": "sum",
    "Order_ID": "count"
}).reset_index()

,Date,Quantity,Sales,Profit,Order_ID
0,2025-07-01,3,3208.35,894.40,2
1,2025-07-02,6,5892.57,1102.89,3
2,2025-07-03,11,12219.41,3502.15,5
3,2025-07-04,6,9604.52,2271.85,4
4,2025-07-05,4,11088.69,2656.92,2
5,2025-07-06,13,16038.48,3805.25,4
6,2025-07-07,7,9826.33,1258.04,3
7,2025-07-08,9,16300.42,1784.02,4
8,2025-07-09,9,18991.28,3965.51,5
9,2025-07-11,1,2530.37,293.58,1


In [87]:
product_daily = data.groupby(["Date", "Product_Name"]).agg({
    "Sales": "sum",
    "Quantity": "sum",
    "Profit": "sum",
    "Order_ID": "count"
}).reset_index()

product_daily.rename(columns={"Order_ID": "Orders"}, inplace=True)

In [88]:
product_daily.head()

,Date,Product_Name,Sales,Quantity,Profit,Orders
0,2024-01-01,Air Fryer,5249.61,1,968.52,1
1,2024-01-01,Bluetooth Speaker,1319.18,1,73.34,1
2,2024-01-01,Competitive Exam Guide,692.01,2,92.17,1
3,2024-01-01,Electric Kettle,5141.26,5,913.49,1
4,2024-01-01,Fiction Bestseller Bundle,488.81,1,56.40,1


In [89]:
all_dates = data["Date"].unique()
all_products = data["Product_Name"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_dates, all_products],
    names=["Date", "Product_Name"]
)

product_daily = (
    product_daily
    .set_index(["Date", "Product_Name"])
    .reindex(complete_index, fill_value=0)
    .reset_index()
)

In [90]:
product_daily.shape

(14620, 6)

In [91]:
product_daily.head(20)

,Date,Product_Name,Sales,Quantity,Profit,Orders
0,2024-01-01,Yoga Mat,1308.18,2,267.56,1
1,2024-01-01,Kids T-Shirt Pack,4173.50,8,1023.96,2
2,2024-01-01,"Smart LED TV 43""",21984.14,1,1677.59,1
3,2024-01-01,Competitive Exam Guide,692.01,2,92.17,1
4,2024-01-01,Skincare Combo Kit,9008.32,7,2100.59,3
5,2024-01-01,Bluetooth Speaker,1319.18,1,73.34,1
6,2024-01-01,Men's Casual Shirt,3021.92,4,937.25,2
7,2024-01-01,Air Fryer,5249.61,1,968.52,1
8,2024-01-01,Fiction Bestseller Bundle,488.81,1,56.40,1
9,2024-01-01,Perfume 100ml,12036.21,6,2925.19,2


In [92]:
product_activity = product_daily.groupby("Product_Name").agg({
    "Sales": lambda x: (x > 0).sum(),
    "Orders": lambda x: (x > 0).sum()
}).reset_index()

product_activity.rename(columns={
    "Sales": "Active_Sales_Days",
    "Orders": "Active_Order_Days"
}, inplace=True)

product_activity.sort_values("Active_Sales_Days")

,Product_Name,Active_Sales_Days,Active_Order_Days
14,"Smart LED TV 43""",192,192
16,Winter Jacket,271,271
0,Air Fryer,274,274
4,Cricket Bat,298,298
10,Non-stick Cookware Set,318,318
7,Fitness Smartwatch,342,342
12,Running Shoes,388,388
11,Perfume 100ml,395,395
1,Bedsheet Set (King),402,402
8,Kids T-Shirt Pack,420,420


In [93]:
product_daily[
    (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
    (product_daily["Date"].between("2024-04-14", "2024-04-24"))
][["Date", "Sales", "Quantity", "Profit", "Orders"]]

,Date,Sales,Quantity,Profit,Orders
2092,2024-04-14,0.00,0,0.00,0
2112,2024-04-15,2720.70,1,323.72,1
2132,2024-04-16,0.00,0,0.00,0
2152,2024-04-17,2614.57,1,325.67,1
2172,2024-04-18,0.00,0,0.00,0
2192,2024-04-19,0.00,0,0.00,0
2212,2024-04-20,7656.44,3,495.00,2
2232,2024-04-21,5282.95,2,440.50,2
2252,2024-04-22,2354.97,1,357.96,1
2272,2024-04-23,0.00,0,0.00,0


In [94]:
product_daily[
    (product_daily["Product_Name"] == "Running Shoes") &
    (product_daily["Date"].between("2025-05-15", "2025-05-25"))
][["Date", "Sales", "Quantity", "Profit", "Orders"]]

,Date,Sales,Quantity,Profit,Orders
10015,2025-05-15,6215.04,2,1152.83,1
10035,2025-05-16,13608.32,4,2294.26,2
10055,2025-05-17,21367.37,6,5495.49,3
10075,2025-05-18,6736.10,2,1635.84,1
10095,2025-05-19,3304.63,1,550.10,1
10115,2025-05-20,0.00,0,0.00,0
10135,2025-05-21,7647.80,2,1514.78,1
10155,2025-05-22,2993.97,1,654.52,1
10175,2025-05-23,16147.63,4,3690.28,1
10195,2025-05-24,8151.81,2,1956.98,1


In [95]:
product_daily[
    product_daily["Product_Name"].isin(
        ["Wireless Earbuds Pro", "Running Shoes"]
    )
].groupby("Product_Name")["Sales"].describe()

,count,mean,std,min,25%,50%,75%,max
Product_Name,,,,,,,,
Running Shoes,731.0,4833.04922,6122.053737,0.0,0.0,3620.98,7715.810,30880.89
Wireless Earbuds Pro,731.0,5500.90803,5750.728680,0.0,0.0,4507.66,7863.965,42854.54


In [96]:
product_daily["Day"] = product_daily["Date"].dt.day_name()

product_daily["Same_Day_4week_Avg"] = (
    product_daily
    .groupby(["Product_Name", "Day"])["Sales"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [97]:
product_daily["Sales_Change_pct"] = (
    (product_daily["Sales"] - product_daily["Same_Day_4week_Avg"])
    / product_daily["Same_Day_4week_Avg"]
) * 100

In [98]:
product_daily[
    (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
    (product_daily["Date"].between("2024-04-14", "2024-04-24"))
][[
    "Date",
    "Sales",
    "Quantity",
    "Same_Day_4week_Avg",
    "Sales_Change_pct"
]]

,Date,Sales,Quantity,Same_Day_4week_Avg,Sales_Change_pct
2092,2024-04-14,0.00,0,4288.4100,-100.000000
2112,2024-04-15,2720.70,1,2804.5225,-2.988833
2132,2024-04-16,0.00,0,604.8675,-100.000000
2152,2024-04-17,2614.57,1,2484.8125,5.222024
2172,2024-04-18,0.00,0,0.0000,NaN
2192,2024-04-19,0.00,0,7494.5350,-100.000000
2212,2024-04-20,7656.44,3,6239.0150,22.718730
2232,2024-04-21,5282.95,2,3656.6775,44.474048
2252,2024-04-22,2354.97,1,1270.6025,85.342780
2272,2024-04-23,0.00,0,0.0000,NaN


In [99]:
product_daily[
    (product_daily["Product_Name"] == "Running Shoes") &
    (product_daily["Date"].between("2025-05-15", "2025-05-25"))
][[
    "Date",
    "Sales",
    "Quantity",
    "Same_Day_4week_Avg",
    "Sales_Change_pct"
]]

,Date,Sales,Quantity,Same_Day_4week_Avg,Sales_Change_pct
10015,2025-05-15,6215.04,2,1038.9975,498.176608
10035,2025-05-16,13608.32,4,3887.2450,250.076211
10055,2025-05-17,21367.37,6,5026.8975,325.060786
10075,2025-05-18,6736.10,2,932.7250,622.195717
10095,2025-05-19,3304.63,1,4632.9425,-28.671034
10115,2025-05-20,0.00,0,3888.1650,-100.000000
10135,2025-05-21,7647.80,2,963.0750,694.102225
10155,2025-05-22,2993.97,1,1553.7600,92.691922
10175,2025-05-23,16147.63,4,7289.3250,121.524352
10195,2025-05-24,8151.81,2,10368.7400,-21.380901


In [100]:
product_daily["Quantity_4week_Avg"] = (
    product_daily
    .groupby(["Product_Name", "Day"])["Quantity"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

product_daily["Quantity_Change_pct"] = (
    (product_daily["Quantity"] - product_daily["Quantity_4week_Avg"])
    / product_daily["Quantity_4week_Avg"]
) * 100

In [101]:
product_daily[
    (product_daily["Product_Name"].isin(
        ["Wireless Earbuds Pro", "Running Shoes"]
    )) &
    (
        product_daily["Date"].between("2024-04-14", "2024-04-24") |
        product_daily["Date"].between("2025-05-15", "2025-05-25")
    )
][[
    "Date",
    "Product_Name",
    "Quantity",
    "Quantity_4week_Avg",
    "Quantity_Change_pct"
]]

,Date,Product_Name,Quantity,Quantity_4week_Avg,Quantity_Change_pct
2092,2024-04-14,Wireless Earbuds Pro,0,1.75,-100.000000
2095,2024-04-14,Running Shoes,0,3.00,-100.000000
2112,2024-04-15,Wireless Earbuds Pro,1,1.25,-20.000000
2115,2024-04-15,Running Shoes,0,0.00,NaN
2132,2024-04-16,Wireless Earbuds Pro,0,0.25,-100.000000
2135,2024-04-16,Running Shoes,2,0.50,300.000000
2152,2024-04-17,Wireless Earbuds Pro,1,1.00,0.000000
2155,2024-04-17,Running Shoes,0,2.00,-100.000000
2172,2024-04-18,Wireless Earbuds Pro,0,0.00,NaN
2175,2024-04-18,Running Shoes,0,2.00,-100.000000


In [102]:
def detect_product_drop(row):
    if (
        pd.isna(row["Sales_Change_pct"]) or
        pd.isna(row["Quantity_Change_pct"])
    ):
        return "Insufficient_Baseline"

    if (
        row["Sales_Change_pct"] <= -30 and
        row["Quantity_Change_pct"] <= -30
    ):
        return "Drop"

    return "Normal"


product_daily["Product_Anomaly"] = product_daily.apply(
    detect_product_drop,
    axis=1
)

In [103]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Product_Anomaly"
]]

,Date,Product_Name,Sales_Change_pct,Quantity_Change_pct,Product_Anomaly
2092,2024-04-14,Wireless Earbuds Pro,-100.000000,-100.000000,Drop
2112,2024-04-15,Wireless Earbuds Pro,-2.988833,-20.000000,Normal
2132,2024-04-16,Wireless Earbuds Pro,-100.000000,-100.000000,Drop
2152,2024-04-17,Wireless Earbuds Pro,5.222024,0.000000,Normal
2172,2024-04-18,Wireless Earbuds Pro,NaN,NaN,Insufficient_Baseline
2192,2024-04-19,Wireless Earbuds Pro,-100.000000,-100.000000,Drop
2212,2024-04-20,Wireless Earbuds Pro,22.718730,20.000000,Normal
2232,2024-04-21,Wireless Earbuds Pro,44.474048,33.333333,Normal
2252,2024-04-22,Wireless Earbuds Pro,85.342780,100.000000,Normal
2272,2024-04-23,Wireless Earbuds Pro,NaN,NaN,Insufficient_Baseline


In [104]:
product_daily["Drop_Signal"] = (
    (product_daily["Sales_Change_pct"] <= -30) &
    (product_daily["Quantity_Change_pct"] <= -30)
)

In [105]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Drop_Signal"
]]

,Date,Product_Name,Sales_Change_pct,Quantity_Change_pct,Drop_Signal
2092,2024-04-14,Wireless Earbuds Pro,-100.000000,-100.000000,True
2112,2024-04-15,Wireless Earbuds Pro,-2.988833,-20.000000,False
2132,2024-04-16,Wireless Earbuds Pro,-100.000000,-100.000000,True
2152,2024-04-17,Wireless Earbuds Pro,5.222024,0.000000,False
2172,2024-04-18,Wireless Earbuds Pro,NaN,NaN,False
2192,2024-04-19,Wireless Earbuds Pro,-100.000000,-100.000000,True
2212,2024-04-20,Wireless Earbuds Pro,22.718730,20.000000,False
2232,2024-04-21,Wireless Earbuds Pro,44.474048,33.333333,False
2252,2024-04-22,Wireless Earbuds Pro,85.342780,100.000000,False
2272,2024-04-23,Wireless Earbuds Pro,NaN,NaN,False


In [106]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Sales",
    "Same_Day_4week_Avg",
    "Quantity",
    "Quantity_4week_Avg"
]]

,Date,Product_Name,Sales,Same_Day_4week_Avg,Quantity,Quantity_4week_Avg
2092,2024-04-14,Wireless Earbuds Pro,0.00,4288.4100,0,1.75
2112,2024-04-15,Wireless Earbuds Pro,2720.70,2804.5225,1,1.25
2132,2024-04-16,Wireless Earbuds Pro,0.00,604.8675,0,0.25
2152,2024-04-17,Wireless Earbuds Pro,2614.57,2484.8125,1,1.00
2172,2024-04-18,Wireless Earbuds Pro,0.00,0.0000,0,0.00
2192,2024-04-19,Wireless Earbuds Pro,0.00,7494.5350,0,3.00
2212,2024-04-20,Wireless Earbuds Pro,7656.44,6239.0150,3,2.50
2232,2024-04-21,Wireless Earbuds Pro,5282.95,3656.6775,2,1.50
2252,2024-04-22,Wireless Earbuds Pro,2354.97,1270.6025,1,0.50
2272,2024-04-23,Wireless Earbuds Pro,0.00,0.0000,0,0.00


In [107]:
product_daily["Sales_Deviation"] = (
    product_daily["Sales"] - product_daily["Same_Day_4week_Avg"]
)

product_daily["Quantity_Deviation"] = (
    product_daily["Quantity"] - product_daily["Quantity_4week_Avg"]
)

In [108]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Sales_Deviation",
    "Quantity_Deviation"
]]

,Date,Product_Name,Sales_Deviation,Quantity_Deviation
2092,2024-04-14,Wireless Earbuds Pro,-4288.4100,-1.75
2112,2024-04-15,Wireless Earbuds Pro,-83.8225,-0.25
2132,2024-04-16,Wireless Earbuds Pro,-604.8675,-0.25
2152,2024-04-17,Wireless Earbuds Pro,129.7575,0.00
2172,2024-04-18,Wireless Earbuds Pro,0.0000,0.00
2192,2024-04-19,Wireless Earbuds Pro,-7494.5350,-3.00
2212,2024-04-20,Wireless Earbuds Pro,1417.4250,0.50
2232,2024-04-21,Wireless Earbuds Pro,1626.2725,0.50
2252,2024-04-22,Wireless Earbuds Pro,1084.3675,0.50
2272,2024-04-23,Wireless Earbuds Pro,0.0000,0.00


In [109]:
product_daily["Same_Day_4week_Avg"].describe()

count    14060.000000
mean      3325.720230
std       3910.941512
min          0.000000
25%       1191.481875
50%       2406.466250
75%       4302.559375
max      80460.672500
Name: Same_Day_4week_Avg, dtype: float64

In [110]:
product_daily["Quantity_4week_Avg"].describe()

count    14060.000000
mean         1.783944
std          1.415159
min          0.000000
25%          0.750000
50%          1.500000
75%          2.500000
max          8.750000
Name: Quantity_4week_Avg, dtype: float64

In [111]:
product_daily["Valid_Baseline"] = (
    (product_daily["Same_Day_4week_Avg"] >= 1000) &
    (product_daily["Quantity_4week_Avg"] >= 1)
)

In [112]:
product_daily[["Same_Day_4week_Avg", "Quantity_4week_Avg", "Valid_Baseline"]].head()

,Same_Day_4week_Avg,Quantity_4week_Avg,Valid_Baseline
0,NaN,NaN,False
1,NaN,NaN,False
2,NaN,NaN,False
3,NaN,NaN,False
4,NaN,NaN,False


In [113]:
product_daily["Product_Drop"] = (
    product_daily["Valid_Baseline"] &
    (product_daily["Sales_Change_pct"] <= -30) &
    (product_daily["Quantity_Change_pct"] <= -30)
)

In [114]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Same_Day_4week_Avg",
    "Quantity_4week_Avg",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Valid_Baseline",
    "Product_Drop"
]]

,Date,Product_Name,Same_Day_4week_Avg,Quantity_4week_Avg,Sales_Change_pct,Quantity_Change_pct,Valid_Baseline,Product_Drop
2092,2024-04-14,Wireless Earbuds Pro,4288.4100,1.75,-100.000000,-100.000000,True,True
2112,2024-04-15,Wireless Earbuds Pro,2804.5225,1.25,-2.988833,-20.000000,True,False
2132,2024-04-16,Wireless Earbuds Pro,604.8675,0.25,-100.000000,-100.000000,False,False
2152,2024-04-17,Wireless Earbuds Pro,2484.8125,1.00,5.222024,0.000000,True,False
2172,2024-04-18,Wireless Earbuds Pro,0.0000,0.00,NaN,NaN,False,False
2192,2024-04-19,Wireless Earbuds Pro,7494.5350,3.00,-100.000000,-100.000000,True,True
2212,2024-04-20,Wireless Earbuds Pro,6239.0150,2.50,22.718730,20.000000,True,False
2232,2024-04-21,Wireless Earbuds Pro,3656.6775,1.50,44.474048,33.333333,True,False
2252,2024-04-22,Wireless Earbuds Pro,1270.6025,0.50,85.342780,100.000000,False,False
2272,2024-04-23,Wireless Earbuds Pro,0.0000,0.00,NaN,NaN,False,False


In [115]:
product_daily["Product_Drop"].value_counts()

Product_Drop
False    9724
True     4896
Name: count, dtype: int64

In [116]:
product_daily["Product_Drop"].mean() * 100

np.float64(33.48837209302326)

In [117]:
product_daily.groupby("Product_Name")["Product_Drop"].mean().sort_values(ascending=False)

Product_Name
Electric Kettle                    0.426813
Skincare Combo Kit                 0.424077
Wireless Earbuds Pro               0.421341
Women's Kurti                      0.419973
Smartphone Case & Charger Combo    0.417237
Men's Casual Shirt                 0.417237
Bedsheet Set (King)                0.403557
Bluetooth Speaker                  0.391245
Yoga Mat                           0.384405
Fiction Bestseller Bundle          0.361149
Running Shoes                      0.351573
Fitness Smartwatch                 0.337893
Perfume 100ml                      0.329685
Kids T-Shirt Pack                  0.320109
Non-stick Cookware Set             0.299590
Cricket Bat                        0.270862
Winter Jacket                      0.231190
Competitive Exam Guide             0.225718
Air Fryer                          0.172367
Smart LED TV 43"                   0.091655
Name: Product_Drop, dtype: float64

In [118]:
product_daily[
    product_daily["Product_Drop"]
][[
    "Product_Name",
    "Sales_Deviation",
    "Quantity_Deviation"
]].describe()

,Sales_Deviation,Quantity_Deviation
count,4896.000000,4896.000000
mean,-3914.043041,-2.146548
std,4521.401078,1.168806
min,-80460.672500,-8.750000
25%,-4662.665625,-2.750000
50%,-2939.091250,-1.750000
75%,-1794.531875,-1.250000
max,-349.620000,-0.500000


In [119]:
product_daily[
    product_daily["Product_Drop"]
][[
    "Sales_Deviation",
    "Quantity_Deviation"
]].quantile([0.25, 0.5, 0.75, 0.9, 0.95])

,Sales_Deviation,Quantity_Deviation
0.25,-4662.665625,-2.75
0.50,-2939.091250,-1.75
0.75,-1794.531875,-1.25
0.90,-1182.341250,-1.00
0.95,-962.430625,-0.75


In [120]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-18", "2024-04-20"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-19", "2025-05-21"))
    )
][[
    "Date",
    "Product_Name",
    "Sales",
    "Same_Day_4week_Avg",
    "Sales_Deviation",
    "Quantity",
    "Quantity_4week_Avg",
    "Quantity_Deviation"
]]

,Date,Product_Name,Sales,Same_Day_4week_Avg,Sales_Deviation,Quantity,Quantity_4week_Avg,Quantity_Deviation
2172,2024-04-18,Wireless Earbuds Pro,0.00,0.0000,0.0000,0,0.00,0.00
2192,2024-04-19,Wireless Earbuds Pro,0.00,7494.5350,-7494.5350,0,3.00,-3.00
2212,2024-04-20,Wireless Earbuds Pro,7656.44,6239.0150,1417.4250,3,2.50,0.50
10095,2025-05-19,Running Shoes,3304.63,4632.9425,-1328.3125,1,1.25,-0.25
10115,2025-05-20,Running Shoes,0.00,3888.1650,-3888.1650,0,1.00,-1.00
10135,2025-05-21,Running Shoes,7647.80,963.0750,6684.7250,2,0.25,1.75


In [121]:
product_daily["Rolling_Dev_Mean"] = (
    product_daily
    .groupby("Product_Name")["Sales_Deviation"]
    .transform(lambda x: x.rolling(30).mean().shift(1))
)

product_daily["Rolling_Dev_Std"] = (
    product_daily
    .groupby("Product_Name")["Sales_Deviation"]
    .transform(lambda x: x.rolling(30).std().shift(1))
)

product_daily["Sales_Deviation_Z"] = (
    (product_daily["Sales_Deviation"] - product_daily["Rolling_Dev_Mean"])
    / product_daily["Rolling_Dev_Std"]
)

In [122]:
product_daily[
    (
        (product_daily["Product_Name"] == "Wireless Earbuds Pro") &
        (product_daily["Date"].between("2024-04-14", "2024-04-24"))
    )
    |
    (
        (product_daily["Product_Name"] == "Running Shoes") &
        (product_daily["Date"].between("2025-05-15", "2025-05-25"))
    )
][[
    "Date",
    "Product_Name",
    "Sales_Deviation",
    "Rolling_Dev_Mean",
    "Rolling_Dev_Std",
    "Sales_Deviation_Z"
]]

,Date,Product_Name,Sales_Deviation,Rolling_Dev_Mean,Rolling_Dev_Std,Sales_Deviation_Z
2092,2024-04-14,Wireless Earbuds Pro,-4288.4100,-1651.547333,4616.731593,-0.571154
2112,2024-04-15,Wireless Earbuds Pro,-83.8225,-1874.085500,4576.038240,0.391226
2132,2024-04-16,Wireless Earbuds Pro,-604.8675,-1644.446083,4483.255002,0.231880
2152,2024-04-17,Wireless Earbuds Pro,129.7575,-1456.129833,4400.835579,0.360361
2172,2024-04-18,Wireless Earbuds Pro,0.0000,-1528.731833,4354.319566,0.351084
2192,2024-04-19,Wireless Earbuds Pro,-7494.5350,-1550.381417,4344.720094,-1.368133
2212,2024-04-20,Wireless Earbuds Pro,1417.4250,-1805.168750,4464.103826,0.721890
2232,2024-04-21,Wireless Earbuds Pro,1626.2725,-1543.775333,4413.602374,0.718245
2252,2024-04-22,Wireless Earbuds Pro,1084.3675,-1730.267417,4140.217452,0.679828
2272,2024-04-23,Wireless Earbuds Pro,0.0000,-1738.347083,4134.278179,0.420472


In [123]:
ground_truth

,Anomaly_ID,Start_Date,End_Date,Anomaly_Type,Affected_Product,Affected_Category,Affected_Region,Expected_Impact
0,A001,2024-02-10,2024-02-12,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 28%...
1,A002,2024-03-05,2024-03-08,Regional Sales Drop,All,All,West,Sales and Quantity decreased approximately 32%...
2,A003,2024-04-18,2024-04-20,Product Demand Decline,Wireless Earbuds Pro,Electronics,All,Sales and Quantity decreased approximately 42%...
3,A004,2024-05-22,2024-05-25,Profit Margin Drop,All,Electronics,All,Sales stayed within the normal range; Profit d...
4,A005,2024-07-14,2024-07-15,Unusual Sales Spike,All,Sports,All,Sales and order volume increased approximately...
5,A006,2024-08-27,2024-08-30,"Margin Problem (Sales Up, Profit Down)",All,Clothing,All,Sales increased approximately 15% for the affe...
6,A007,2024-10-05,2024-10-07,Regional Sales Drop,All,All,South,Sales and Quantity decreased approximately 29%...
7,A008,2024-11-11,2024-11-12,Product Sales Spike,Air Fryer,Home & Kitchen,All,Sales and order volume increased approximately...
8,A009,2025-01-09,2025-01-13,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 24%...
9,A010,2025-03-03,2025-03-05,Profit Margin Drop,All,Home & Kitchen,All,Sales stayed within the normal range; Profit d...


In [124]:
daily_data = data.groupby("Date").agg({
    "Quantity": "sum",
    "Sales": "sum",
    "Profit": "sum",
    "Order_ID": "count"
}).reset_index()

daily_data.rename(columns={"Order_ID": "Orders"}, inplace=True)

In [125]:
daily_data["Day"] = daily_data["Date"].dt.day_name()

daily_data["Sales_4week_Avg"] = (
    daily_data
    .groupby("Day")["Sales"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

daily_data["Quantity_4week_Avg"] = (
    daily_data
    .groupby("Day")["Quantity"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [126]:
daily_data["Sales_Change_pct"] = (
    (daily_data["Sales"] - daily_data["Sales_4week_Avg"])
    / daily_data["Sales_4week_Avg"]
) * 100

daily_data["Quantity_Change_pct"] = (
    (daily_data["Quantity"] - daily_data["Quantity_4week_Avg"])
    / daily_data["Quantity_4week_Avg"]
) * 100

In [127]:
daily_data[
    daily_data["Date"].between("2024-02-06", "2024-02-16")
][[
    "Date",
    "Sales",
    "Sales_4week_Avg",
    "Sales_Change_pct",
    "Quantity",
    "Quantity_4week_Avg",
    "Quantity_Change_pct"
]]

,Date,Sales,Sales_4week_Avg,Sales_Change_pct,Quantity,Quantity_4week_Avg,Quantity_Change_pct
36,2024-02-06,36341.30,33697.4425,7.845870,41,20.25,102.469136
37,2024-02-07,69369.84,59993.5625,15.628806,32,28.25,13.274336
38,2024-02-08,56369.41,73066.0625,-22.851447,33,35.75,-7.692308
39,2024-02-09,44466.47,70784.3625,-37.180377,20,35.75,-44.055944
40,2024-02-10,27856.27,73015.3250,-61.848735,18,39.25,-54.140127
41,2024-02-11,16940.17,57132.9975,-70.349587,13,33.75,-61.481481
42,2024-02-12,32755.84,36160.8350,-9.416251,20,23.00,-13.043478
43,2024-02-13,27512.15,34992.2825,-21.376521,19,25.75,-26.213592
44,2024-02-14,45244.45,62005.4275,-27.031468,28,28.25,-0.884956
45,2024-02-15,30247.34,56401.1950,-46.371101,18,30.75,-41.463415


In [128]:
overall_drop_signal = (
    (daily_data["Sales_Change_pct"] <= -30) &
    (daily_data["Quantity_Change_pct"] <= -30)
)

overall_drop_signal.mean() * 100

np.float64(11.49110807113543)

In [129]:
overall_drop_signal.value_counts()

False    647
True      84
Name: count, dtype: int64

In [130]:
daily_data["Overall_Drop_Signal"] = (
    (daily_data["Sales_Change_pct"] <= -30) &
    (daily_data["Quantity_Change_pct"] <= -30)
)

In [131]:
daily_data["Drop_Group"] = (
    daily_data["Overall_Drop_Signal"] !=
    daily_data["Overall_Drop_Signal"].shift()
).cumsum()

In [132]:
drop_runs = (
    daily_data[daily_data["Overall_Drop_Signal"]]
    .groupby("Drop_Group")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Duration=("Date", "count")
    )
    .sort_values("Duration", ascending=False)
)

drop_runs

,Start_Date,End_Date,Duration
Drop_Group,,,
6,2024-02-09,2024-02-11,3
68,2024-11-18,2024-11-20,3
144,2025-12-29,2025-12-30,2
18,2024-03-21,2024-03-22,2
96,2025-04-14,2025-04-15,2
...,...,...,...
66,2024-11-16,2024-11-16,1
70,2024-11-22,2024-11-22,1
72,2024-12-01,2024-12-01,1


In [133]:
drop_runs = (
    daily_data[daily_data["Overall_Drop_Signal"]]
    .groupby("Drop_Group")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Duration=("Date", "count"),
        Avg_Sales_Change=("Sales_Change_pct", "mean"),
        Avg_Quantity_Change=("Quantity_Change_pct", "mean")
    )
    .sort_values(
        ["Duration", "Avg_Sales_Change"],
        ascending=[False, True]
    )
)

drop_runs

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change
Drop_Group,,,,,
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851
68,2024-11-18,2024-11-20,3,-42.502709,-45.753320
64,2024-11-13,2024-11-14,2,-67.837573,-54.815553
144,2025-12-29,2025-12-30,2,-67.664705,-60.156339
4,2024-02-02,2024-02-03,2,-62.962044,-50.463130
...,...,...,...,...,...
34,2024-05-26,2024-05-26,1,-37.641558,-44.508671
126,2025-09-22,2025-09-22,1,-35.986548,-58.041958
100,2025-05-10,2025-05-10,1,-34.409730,-33.333333


In [134]:
drop_runs["Severity"] = (
    (-drop_runs["Avg_Sales_Change"]) +
    (-drop_runs["Avg_Quantity_Change"])
) / 2

In [135]:
drop_runs.sort_values("Severity", ascending=False).head(15)

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Severity
Drop_Group,,,,,,
56,2024-09-17,2024-09-17,1,-86.371400,-76.859504,81.615452
88,2025-02-18,2025-02-18,1,-86.602251,-68.000000,77.301126
12,2024-02-20,2024-02-20,1,-73.483622,-66.666667,70.075144
106,2025-05-29,2025-05-29,1,-68.536063,-71.084337,69.810200
36,2024-06-05,2024-06-05,1,-62.384177,-74.803150,68.593663
44,2024-07-27,2024-07-27,1,-67.919780,-61.363636,64.641708
144,2025-12-29,2025-12-30,2,-67.664705,-60.156339,63.910522
22,2024-04-03,2024-04-03,1,-62.239099,-64.444444,63.341771
128,2025-09-29,2025-09-29,1,-73.783476,-50.387597,62.085536


In [136]:
daily_data[
    (daily_data["Date"] >= "2025-01-07") &
    (daily_data["Date"] <= "2025-01-15")
][[
    "Date",
    "Sales",
    "Sales_4week_Avg",
    "Sales_Change_pct",
    "Quantity",
    "Quantity_4week_Avg",
    "Quantity_Change_pct",
    "Overall_Drop_Signal"
]]

,Date,Sales,Sales_4week_Avg,Sales_Change_pct,Quantity,Quantity_4week_Avg,Quantity_Change_pct,Overall_Drop_Signal
372,2025-01-07,92134.18,67806.8375,35.877418,34,37.50,-9.333333,False
373,2025-01-08,53873.48,68775.7025,-21.667859,44,47.25,-6.878307,False
374,2025-01-09,42037.84,70765.9375,-40.595940,32,39.25,-18.471338,False
375,2025-01-10,44718.96,59140.5500,-24.385282,25,51.25,-51.219512,False
376,2025-01-11,56159.27,97741.8975,-42.543299,35,44.50,-21.348315,False
377,2025-01-12,42161.54,102319.9475,-58.794408,21,55.75,-62.331839,True
378,2025-01-13,51640.09,49837.4125,3.617117,16,33.75,-52.592593,False
379,2025-01-14,76528.10,77016.8900,-0.634653,31,38.25,-18.954248,False
380,2025-01-15,52535.98,64674.9500,-18.769199,34,52.25,-34.928230,False


In [137]:
daily_data["Overall_Drop_Signal_Soft"] = (
    (daily_data["Sales_Change_pct"] <= -30) |
    (daily_data["Quantity_Change_pct"] <= -30)
)

In [138]:
daily_data[
    (
        (daily_data["Date"] >= "2024-02-07") &
        (daily_data["Date"] <= "2024-02-15")
    ) |
    (
        (daily_data["Date"] >= "2025-01-07") &
        (daily_data["Date"] <= "2025-01-15")
    )
][[
    "Date",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Overall_Drop_Signal_Soft"
]]

,Date,Sales_Change_pct,Quantity_Change_pct,Overall_Drop_Signal_Soft
37,2024-02-07,15.628806,13.274336,False
38,2024-02-08,-22.851447,-7.692308,False
39,2024-02-09,-37.180377,-44.055944,True
40,2024-02-10,-61.848735,-54.140127,True
41,2024-02-11,-70.349587,-61.481481,True
42,2024-02-12,-9.416251,-13.043478,False
43,2024-02-13,-21.376521,-26.213592,False
44,2024-02-14,-27.031468,-0.884956,False
45,2024-02-15,-46.371101,-41.463415,True
372,2025-01-07,35.877418,-9.333333,False


In [139]:
daily_data["Soft_Drop_Group"] = (
    daily_data["Overall_Drop_Signal_Soft"] !=
    daily_data["Overall_Drop_Signal_Soft"].shift()
).cumsum()

In [140]:
soft_drop_runs = (
    daily_data[daily_data["Overall_Drop_Signal_Soft"]]
    .groupby("Soft_Drop_Group")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Duration=("Date", "count"),
        Avg_Sales_Change=("Sales_Change_pct", "mean"),
        Avg_Quantity_Change=("Quantity_Change_pct", "mean")
    )
    .sort_values(
        ["Duration", "Start_Date"],
        ascending=[False, True]
    )
)

soft_drop_runs

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change
Soft_Drop_Group,,,,,
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356
...,...,...,...,...,...
256,2025-11-21,2025-11-21,1,-49.680543,-22.580645
258,2025-11-28,2025-11-28,1,-49.747885,-14.666667
260,2025-12-02,2025-12-02,1,-49.585479,-23.456790


In [141]:
candidate_events = soft_drop_runs[
    (soft_drop_runs["Duration"] >= 3) &
    (soft_drop_runs["Avg_Sales_Change"] <= -30)
].copy()

candidate_events

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change
Soft_Drop_Group,,,,,
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140
180,2025-05-08,2025-05-11,4,-36.682387,-16.107833
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851
32,2024-04-07,2024-04-09,3,-38.592093,-38.089509


In [142]:
evaluation = pd.DataFrame({
    "Event": ["A001", "A009"],
    "Ground_Truth_Start": ["2024-02-10", "2025-01-09"],
    "Ground_Truth_End": ["2024-02-12", "2025-01-13"],
    "Detected": [True, True]
})

evaluation

,Event,Ground_Truth_Start,Ground_Truth_End,Detected
0,A001,2024-02-10,2024-02-12,True
1,A009,2025-01-09,2025-01-13,True


In [143]:
detection_rate = evaluation["Detected"].mean() * 100
detection_rate

np.float64(100.0)

In [144]:
ground_truth = pd.read_excel(
    "ecommerce_business_monitoring_dataset.xlsx",
    sheet_name="Ground_Truth"
)

ground_truth

,Anomaly_ID,Start_Date,End_Date,Anomaly_Type,Affected_Product,Affected_Category,Affected_Region,Expected_Impact
0,A001,2024-02-10,2024-02-12,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 28%...
1,A002,2024-03-05,2024-03-08,Regional Sales Drop,All,All,West,Sales and Quantity decreased approximately 32%...
2,A003,2024-04-18,2024-04-20,Product Demand Decline,Wireless Earbuds Pro,Electronics,All,Sales and Quantity decreased approximately 42%...
3,A004,2024-05-22,2024-05-25,Profit Margin Drop,All,Electronics,All,Sales stayed within the normal range; Profit d...
4,A005,2024-07-14,2024-07-15,Unusual Sales Spike,All,Sports,All,Sales and order volume increased approximately...
5,A006,2024-08-27,2024-08-30,"Margin Problem (Sales Up, Profit Down)",All,Clothing,All,Sales increased approximately 15% for the affe...
6,A007,2024-10-05,2024-10-07,Regional Sales Drop,All,All,South,Sales and Quantity decreased approximately 29%...
7,A008,2024-11-11,2024-11-12,Product Sales Spike,Air Fryer,Home & Kitchen,All,Sales and order volume increased approximately...
8,A009,2025-01-09,2025-01-13,Overall Sales Drop,All,All,All,Sales and Quantity decreased approximately 24%...
9,A010,2025-03-03,2025-03-05,Profit Margin Drop,All,Home & Kitchen,All,Sales stayed within the normal range; Profit d...


In [145]:
overall_truth = ground_truth[
    ground_truth["Anomaly_Type"] == "Overall Sales Drop"
].copy()

candidate_events["Matched_Event"] = None

for idx, candidate in candidate_events.iterrows():

    for _, truth in overall_truth.iterrows():

        if (
            candidate["Start_Date"] <= truth["End_Date"] and
            candidate["End_Date"] >= truth["Start_Date"]
        ):
            candidate_events.loc[idx, "Matched_Event"] = truth["Anomaly_ID"]

candidate_events[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Avg_Quantity_Change",
        "Matched_Event"
    ]
]

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Matched_Event
Soft_Drop_Group,,,,,,
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482,None
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021,None
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719,A009
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448,None
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356,None
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140,None
180,2025-05-08,2025-05-11,4,-36.682387,-16.107833,None
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851,A001
32,2024-04-07,2024-04-09,3,-38.592093,-38.089509,None


In [146]:
candidate_events["Matched_Event"].notna().value_counts()

Matched_Event
False    11
True      2
Name: count, dtype: int64

In [147]:
false_positives = candidate_events[
    candidate_events["Matched_Event"].isna()
].copy()

false_positives[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Avg_Quantity_Change"
    ]
]

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change
Soft_Drop_Group,,,,,
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140
180,2025-05-08,2025-05-11,4,-36.682387,-16.107833
32,2024-04-07,2024-04-09,3,-38.592093,-38.089509
102,2024-09-30,2024-10-02,3,-32.448439,-27.863291
122,2024-11-26,2024-11-28,3,-50.609002,-13.871982


In [148]:
daily_data[
    (daily_data["Date"] >= "2024-01-20") &
    (daily_data["Date"] <= "2024-02-10")
][[
    "Date",
    "Sales",
    "Sales_4week_Avg",
    "Sales_Change_pct",
    "Quantity",
    "Quantity_4week_Avg",
    "Quantity_Change_pct"
]]

,Date,Sales,Sales_4week_Avg,Sales_Change_pct,Quantity,Quantity_4week_Avg,Quantity_Change_pct
19,2024-01-20,66303.94,NaN,NaN,43,NaN,NaN
20,2024-01-21,43750.61,NaN,NaN,27,NaN,NaN
21,2024-01-22,35185.77,NaN,NaN,27,NaN,NaN
22,2024-01-23,40955.84,NaN,NaN,19,NaN,NaN
23,2024-01-24,22146.40,NaN,NaN,17,NaN,NaN
24,2024-01-25,89110.51,NaN,NaN,38,NaN,NaN
25,2024-01-26,53440.82,NaN,NaN,37,NaN,NaN
26,2024-01-27,95384.69,NaN,NaN,52,NaN,NaN
27,2024-01-28,67820.32,NaN,NaN,44,NaN,NaN
28,2024-01-29,23554.77,55380.7225,-57.467565,19,30.50,-37.704918


In [149]:
daily_data["Sales_4week_Median"] = (
    daily_data.groupby("Day")["Sales"]
    .transform(lambda x: x.shift(1).rolling(4).median())
)

In [150]:
daily_data["Sales_Median_Change_pct"] = (
    (daily_data["Sales"] - daily_data["Sales_4week_Median"])
    / daily_data["Sales_4week_Median"]
) * 100

In [151]:
daily_data[
    (daily_data["Date"] >= "2024-01-29") &
    (daily_data["Date"] <= "2024-02-05")
][[
    "Date",
    "Sales",
    "Sales_4week_Avg",
    "Sales_4week_Median",
    "Sales_Change_pct",
    "Sales_Median_Change_pct"
]]

,Date,Sales,Sales_4week_Avg,Sales_4week_Median,Sales_Change_pct,Sales_Median_Change_pct
28,2024-01-29,23554.77,55380.7225,57675.080,-57.467565,-59.159536
29,2024-01-30,44627.86,52550.9550,36058.890,-15.076976,23.763821
30,2024-01-31,95707.56,63514.0100,61060.145,50.687321,56.743093
31,2024-02-01,46835.88,80032.4775,81906.025,-41.478908,-42.817540
32,2024-02-02,33495.30,76310.5475,63811.500,-56.106592,-47.508991
33,2024-02-03,24582.38,81445.7950,80844.315,-69.817496,-69.592939
34,2024-02-04,32260.66,62597.3325,60969.160,-48.463203,-47.086921
35,2024-02-05,33076.53,43522.6750,44006.020,-24.001615,-24.836352


In [152]:
region_daily[
    (region_daily["Date"] >= "2024-02-01") &
    (region_daily["Date"] <= "2024-02-04")
][[
    "Date",
    "Region",
    "Sales",
    "Quantity",
    "Orders"
]].sort_values(["Date", "Region"])

,Date,Region,Sales,Quantity,Orders
157,2024-02-01,Central,6834.04,6,4
159,2024-02-01,East,14711.21,10,4
155,2024-02-01,North,0.00,0,0
158,2024-02-01,South,16038.36,7,3
156,2024-02-01,West,9252.27,6,3
162,2024-02-02,Central,5856.88,6,2
164,2024-02-02,East,7164.40,2,2
160,2024-02-02,North,12977.25,10,6
163,2024-02-02,South,3783.13,1,1
161,2024-02-02,West,3713.64,2,1


In [153]:
region_daily[
    (region_daily["Date"] >= "2024-02-01") &
    (region_daily["Date"] <= "2024-02-04")
][[
    "Date",
    "Region",
    "Sales",
    "Same_Day_4week_Avg",
    "Same_Day_Change_pct"
]].sort_values(["Date", "Region"])

,Date,Region,Sales,Same_Day_4week_Avg,Same_Day_Change_pct
157,2024-02-01,Central,6834.04,8672.2075,-21.196074
159,2024-02-01,East,14711.21,10429.6225,41.052181
155,2024-02-01,North,0.00,14459.9675,-100.000000
158,2024-02-01,South,16038.36,35198.5800,-54.434639
156,2024-02-01,West,9252.27,11272.1000,-17.918844
162,2024-02-02,Central,5856.88,17134.7050,-65.818612
164,2024-02-02,East,7164.40,16963.0650,-57.764708
160,2024-02-02,North,12977.25,21932.7200,-40.831552
163,2024-02-02,South,3783.13,11714.6225,-67.705916
161,2024-02-02,West,3713.64,8565.4350,-56.643883


In [154]:
region_daily["Region_Drop"] = (
    region_daily["Same_Day_Change_pct"] <= -30
)

In [155]:
regional_drop_count = (
    region_daily.groupby("Date")["Region_Drop"]
    .sum()
    .reset_index(name="Regions_Down")
)

In [156]:
daily_data = daily_data.merge(
    regional_drop_count,
    on="Date",
    how="left"
)

In [157]:
daily_data[
    (
        (daily_data["Date"] >= "2024-02-01") &
        (daily_data["Date"] <= "2024-02-12")
    )
][[
    "Date",
    "Sales",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Regions_Down"
]]

,Date,Sales,Sales_Change_pct,Quantity_Change_pct,Regions_Down
31,2024-02-01,46835.88,-41.478908,-17.142857,2
32,2024-02-02,33495.30,-56.106592,-45.098039,5
33,2024-02-03,24582.38,-69.817496,-55.828221,4
34,2024-02-04,32260.66,-48.463203,-7.692308,3
35,2024-02-05,33076.53,-24.001615,14.285714,2
36,2024-02-06,36341.30,7.845870,102.469136,0
37,2024-02-07,69369.84,15.628806,13.274336,1
38,2024-02-08,56369.41,-22.851447,-7.692308,2
39,2024-02-09,44466.47,-37.180377,-44.055944,4
40,2024-02-10,27856.27,-61.848735,-54.140127,3


In [158]:
daily_data[
    (daily_data["Date"] >= "2024-02-01") &
    (daily_data["Date"] <= "2024-02-12")
][[
    "Date",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Regions_Down"
]]

,Date,Sales_Change_pct,Quantity_Change_pct,Regions_Down
31,2024-02-01,-41.478908,-17.142857,2
32,2024-02-02,-56.106592,-45.098039,5
33,2024-02-03,-69.817496,-55.828221,4
34,2024-02-04,-48.463203,-7.692308,3
35,2024-02-05,-24.001615,14.285714,2
36,2024-02-06,7.845870,102.469136,0
37,2024-02-07,15.628806,13.274336,1
38,2024-02-08,-22.851447,-7.692308,2
39,2024-02-09,-37.180377,-44.055944,4
40,2024-02-10,-61.848735,-54.140127,3


In [159]:
daily_data[
    (
        (daily_data["Date"] >= "2024-02-01") &
        (daily_data["Date"] <= "2024-02-04")
    ) |
    (
        (daily_data["Date"] >= "2025-02-01") &
        (daily_data["Date"] <= "2025-02-04")
    )
][[
    "Date",
    "Sales",
    "Quantity"
]]

,Date,Sales,Quantity
31,2024-02-01,46835.88,29
32,2024-02-02,33495.30,21
33,2024-02-03,24582.38,18
34,2024-02-04,32260.66,30
397,2025-02-01,81182.74,48
398,2025-02-02,67684.56,27
399,2025-02-03,27382.06,25
400,2025-02-04,52137.47,31


In [160]:
candidate_events["Avg_Regions_Down"] = candidate_events.apply(
    lambda row: daily_data[
        (daily_data["Date"] >= row["Start_Date"]) &
        (daily_data["Date"] <= row["End_Date"])
    ]["Regions_Down"].mean(),
    axis=1
)

candidate_events[[
    "Start_Date",
    "End_Date",
    "Duration",
    "Avg_Sales_Change",
    "Avg_Quantity_Change",
    "Avg_Regions_Down",
    "Matched_Event"
]].sort_values("Avg_Regions_Down", ascending=False)

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Avg_Regions_Down,Matched_Event
Soft_Drop_Group,,,,,,,
270,2025-12-29,2025-12-31,3,-40.906501,-62.457167,4.000000,None
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851,3.666667,A001
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719,3.600000,A009
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021,3.500000,None
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356,3.500000,None
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448,3.400000,None
122,2024-11-26,2024-11-28,3,-50.609002,-13.871982,3.333333,None
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140,3.250000,None
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482,3.000000,None


In [161]:
candidate_events["Severity_Score"] = (
    (candidate_events["Avg_Sales_Change"] <= -50).astype(int)
    + (candidate_events["Avg_Quantity_Change"] <= -40).astype(int)
    + (candidate_events["Avg_Regions_Down"] >= 3.5).astype(int)
    + (candidate_events["Duration"] >= 4).astype(int)
)

In [162]:
candidate_events[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Avg_Quantity_Change",
        "Avg_Regions_Down",
        "Severity_Score",
        "Matched_Event"
    ]
].sort_values("Severity_Score", ascending=False)

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Avg_Regions_Down,Severity_Score,Matched_Event
Soft_Drop_Group,,,,,,,,
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719,3.600000,3,A009
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356,3.500000,3,None
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851,3.666667,3,A001
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021,3.500000,2,None
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140,3.250000,2,None
270,2025-12-29,2025-12-31,3,-40.906501,-62.457167,4.000000,2,None
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482,3.000000,1,None
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448,3.400000,1,None
180,2025-05-08,2025-05-11,4,-36.682387,-16.107833,3.000000,1,None


In [163]:
candidate_events["Weighted_Score"] = (
    2 * (candidate_events["Avg_Sales_Change"] <= -50).astype(int)
    + (candidate_events["Avg_Quantity_Change"] <= -40).astype(int)
    + (candidate_events["Avg_Regions_Down"] >= 3.5).astype(int)
    + (candidate_events["Duration"] >= 4).astype(int)
)

In [164]:
candidate_events[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Avg_Quantity_Change",
        "Avg_Regions_Down",
        "Severity_Score",
        "Weighted_Score",
        "Matched_Event"
    ]
].sort_values("Weighted_Score", ascending=False)

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Avg_Regions_Down,Severity_Score,Weighted_Score,Matched_Event
Soft_Drop_Group,,,,,,,,,
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356,3.500000,3,4,None
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851,3.666667,3,4,A001
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719,3.600000,3,3,A009
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021,3.500000,2,2,None
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140,3.250000,2,2,None
122,2024-11-26,2024-11-28,3,-50.609002,-13.871982,3.333333,1,2,None
270,2025-12-29,2025-12-31,3,-40.906501,-62.457167,4.000000,2,2,None
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482,3.000000,1,1,None
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448,3.400000,1,1,None


In [166]:
daily_data["Profit_4week_Avg"] = (
    daily_data.groupby("Day")["Profit"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

In [167]:
daily_data["Profit_Change_pct"] = (
    (daily_data["Profit"] - daily_data["Profit_4week_Avg"])
    / daily_data["Profit_4week_Avg"]
) * 100

In [168]:
daily_data[
    (
        (daily_data["Date"] >= "2024-02-07") &
        (daily_data["Date"] <= "2024-02-13")
    ) |
    (
        (daily_data["Date"] >= "2025-01-07") &
        (daily_data["Date"] <= "2025-01-15")
    )
][[
    "Date",
    "Sales_Change_pct",
    "Quantity_Change_pct",
    "Profit_Change_pct"
]]

,Date,Sales_Change_pct,Quantity_Change_pct,Profit_Change_pct
37,2024-02-07,15.628806,13.274336,15.841366
38,2024-02-08,-22.851447,-7.692308,-10.279451
39,2024-02-09,-37.180377,-44.055944,-20.849126
40,2024-02-10,-61.848735,-54.140127,-56.644298
41,2024-02-11,-70.349587,-61.481481,-64.957875
42,2024-02-12,-9.416251,-13.043478,29.414632
43,2024-02-13,-21.376521,-26.213592,-18.563293
372,2025-01-07,35.877418,-9.333333,21.617875
373,2025-01-08,-21.667859,-6.878307,-7.225396
374,2025-01-09,-40.595940,-18.471338,-35.553572


In [169]:
candidate_events["Avg_Profit_Change"] = candidate_events.apply(
    lambda row: daily_data[
        (daily_data["Date"] >= row["Start_Date"]) &
        (daily_data["Date"] <= row["End_Date"])
    ]["Profit_Change_pct"].mean(),
    axis=1
)

In [170]:
candidate_events[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Avg_Quantity_Change",
        "Avg_Profit_Change",
        "Avg_Regions_Down",
        "Weighted_Score",
        "Matched_Event"
    ]
].sort_values("Weighted_Score", ascending=False)

,Start_Date,End_Date,Duration,Avg_Sales_Change,Avg_Quantity_Change,Avg_Profit_Change,Avg_Regions_Down,Weighted_Score,Matched_Event
Soft_Drop_Group,,,,,,,,,
4,2024-02-01,2024-02-04,4,-53.966550,-31.440356,-41.706421,3.500000,4,None
6,2024-02-09,2024-02-11,3,-56.459566,-53.225851,-47.483766,3.666667,4,A001
134,2025-01-09,2025-01-13,5,-32.540362,-41.192719,-30.351603,3.600000,3,A009
264,2025-12-10,2025-12-15,6,-49.546403,-22.556021,-48.292411,3.500000,2,None
56,2024-06-09,2024-06-12,4,-45.331797,-41.878140,-33.811865,3.250000,2,None
122,2024-11-26,2024-11-28,3,-50.609002,-13.871982,-48.098774,3.333333,2,None
270,2025-12-29,2025-12-31,3,-40.906501,-62.457167,-47.086357,4.000000,2,None
118,2024-11-16,2024-11-22,7,-32.492904,-34.942482,-27.681239,3.000000,1,None
200,2025-07-01,2025-07-05,5,-42.244652,-31.084448,-33.425776,3.400000,1,None


In [171]:
candidate_events["Worst_Sales_Change"] = candidate_events.apply(
    lambda row: daily_data[
        (daily_data["Date"] >= row["Start_Date"]) &
        (daily_data["Date"] <= row["End_Date"])
    ]["Sales_Change_pct"].min(),
    axis=1
)

In [172]:
candidate_events["Worst_Profit_Change"] = candidate_events.apply(
    lambda row: daily_data[
        (daily_data["Date"] >= row["Start_Date"]) &
        (daily_data["Date"] <= row["End_Date"])
    ]["Profit_Change_pct"].min(),
    axis=1
)

In [173]:
candidate_events[
    [
        "Start_Date",
        "End_Date",
        "Duration",
        "Avg_Sales_Change",
        "Worst_Sales_Change",
        "Avg_Profit_Change",
        "Worst_Profit_Change",
        "Matched_Event"
    ]
].sort_values("Worst_Sales_Change")

,Start_Date,End_Date,Duration,Avg_Sales_Change,Worst_Sales_Change,Avg_Profit_Change,Worst_Profit_Change,Matched_Event
Soft_Drop_Group,,,,,,,,
270,2025-12-29,2025-12-31,3,-40.906501,-81.286211,-47.086357,-63.343130,None
118,2024-11-16,2024-11-22,7,-32.492904,-71.429683,-27.681239,-51.932619,None
6,2024-02-09,2024-02-11,3,-56.459566,-70.349587,-47.483766,-64.957875,A001
4,2024-02-01,2024-02-04,4,-53.966550,-69.817496,-41.706421,-57.457032,None
102,2024-09-30,2024-10-02,3,-32.448439,-63.563207,-36.310566,-58.855485,None
56,2024-06-09,2024-06-12,4,-45.331797,-61.921471,-33.811865,-53.886701,None
264,2025-12-10,2025-12-15,6,-49.546403,-61.343307,-48.292411,-65.431405,None
122,2024-11-26,2024-11-28,3,-50.609002,-60.195215,-48.098774,-52.899201,None
134,2025-01-09,2025-01-13,5,-32.540362,-58.794408,-30.351603,-36.092768,A009


In [174]:
event_start = "2024-02-09"
event_end = "2024-02-11"

event_products = (
    product_daily[
        (product_daily["Date"] >= event_start) &
        (product_daily["Date"] <= event_end)
    ]
    .groupby("Product_Name")
    .agg(
        Event_Sales=("Sales", "sum"),
        Event_Quantity=("Quantity", "sum"),
        Event_Profit=("Profit", "sum")
    )
    .reset_index()
)

In [175]:
baseline_start = "2024-02-06"
baseline_end = "2024-02-08"

baseline_products = (
    product_daily[
        (product_daily["Date"] >= baseline_start) &
        (product_daily["Date"] <= baseline_end)
    ]
    .groupby("Product_Name")
    .agg(
        Baseline_Sales=("Sales", "sum"),
        Baseline_Quantity=("Quantity", "sum"),
        Baseline_Profit=("Profit", "sum")
    )
    .reset_index()
)

In [176]:
event_products = event_products.merge(
    baseline_products,
    on="Product_Name",
    how="left"
)

In [177]:
event_products["Sales_Change_pct"] = (
    (event_products["Event_Sales"] - event_products["Baseline_Sales"])
    / event_products["Baseline_Sales"]
) * 100

In [178]:
event_products.sort_values(
    "Sales_Change_pct"
).head(10)

,Product_Name,Event_Sales,Event_Quantity,Event_Profit,Baseline_Sales,Baseline_Quantity,Baseline_Profit,Sales_Change_pct
10,Non-stick Cookware Set,0.00,0,0.00,15879.66,5,3523.57,-100.000000
5,Electric Kettle,1314.00,1,344.44,15539.29,13,2723.59,-91.544015
7,Fitness Smartwatch,4433.68,1,489.29,33244.64,7,4568.90,-86.663474
3,Competitive Exam Guide,443.46,1,82.79,2230.05,5,339.16,-80.114347
19,Yoga Mat,1554.98,2,454.78,7452.86,9,1917.70,-79.135795
2,Bluetooth Speaker,3305.43,2,453.22,13575.94,8,1571.68,-75.652294
1,Bedsheet Set (King),2893.41,2,467.40,11380.60,8,2433.42,-74.575945
13,Skincare Combo Kit,3289.44,2,1265.37,9344.43,6,3164.80,-64.797853
6,Fiction Bestseller Bundle,4103.43,7,592.18,8934.03,15,1166.97,-54.069664
8,Kids T-Shirt Pack,2764.57,4,1114.12,5676.08,8,1875.24,-51.294379


In [179]:
event_products["Sales_Loss"] = (
    event_products["Baseline_Sales"]
    - event_products["Event_Sales"]
)

In [180]:
event_products.sort_values(
    "Sales_Loss",
    ascending=False
).head(10)

,Product_Name,Event_Sales,Event_Quantity,Event_Profit,Baseline_Sales,Baseline_Quantity,Baseline_Profit,Sales_Change_pct,Sales_Loss
7,Fitness Smartwatch,4433.68,1,489.29,33244.64,7,4568.90,-86.663474,28810.96
10,Non-stick Cookware Set,0.00,0,0.00,15879.66,5,3523.57,-100.000000,15879.66
5,Electric Kettle,1314.00,1,344.44,15539.29,13,2723.59,-91.544015,14225.29
2,Bluetooth Speaker,3305.43,2,453.22,13575.94,8,1571.68,-75.652294,10270.51
1,Bedsheet Set (King),2893.41,2,467.40,11380.60,8,2433.42,-74.575945,8487.19
13,Skincare Combo Kit,3289.44,2,1265.37,9344.43,6,3164.80,-64.797853,6054.99
19,Yoga Mat,1554.98,2,454.78,7452.86,9,1917.70,-79.135795,5897.88
17,Wireless Earbuds Pro,5243.28,2,242.94,10211.66,4,1374.35,-48.653990,4968.38
6,Fiction Bestseller Bundle,4103.43,7,592.18,8934.03,15,1166.97,-54.069664,4830.60
8,Kids T-Shirt Pack,2764.57,4,1114.12,5676.08,8,1875.24,-51.294379,2911.51


In [181]:
total_sales_loss = event_products["Sales_Loss"].sum()

print(total_sales_loss)

72817.64


In [182]:
event_products["Loss_Contribution_pct"] = (
    event_products["Sales_Loss"] / total_sales_loss
) * 100

In [183]:
event_products[
    [
        "Product_Name",
        "Sales_Loss",
        "Loss_Contribution_pct"
    ]
].sort_values(
    "Loss_Contribution_pct",
    ascending=False
).head(10)

,Product_Name,Sales_Loss,Loss_Contribution_pct
7,Fitness Smartwatch,28810.96,39.565907
10,Non-stick Cookware Set,15879.66,21.807436
5,Electric Kettle,14225.29,19.535500
2,Bluetooth Speaker,10270.51,14.104426
1,Bedsheet Set (King),8487.19,11.655404
13,Skincare Combo Kit,6054.99,8.315279
19,Yoga Mat,5897.88,8.099521
17,Wireless Earbuds Pro,4968.38,6.823045
6,Fiction Bestseller Bundle,4830.60,6.633832
8,Kids T-Shirt Pack,2911.51,3.998358


In [184]:
loss_contributors = event_products[
    event_products["Sales_Loss"] > 0
].copy()

In [185]:
total_positive_loss = loss_contributors["Sales_Loss"].sum()

loss_contributors["Loss_Contribution_pct"] = (
    loss_contributors["Sales_Loss"] / total_positive_loss
) * 100

In [187]:
loss_contributors = loss_contributors.sort_values(
    "Loss_Contribution_pct",
    ascending=False
)

loss_contributors[
    ["Product_Name", "Sales_Loss", "Loss_Contribution_pct"]
].head(10)

,Product_Name,Sales_Loss,Loss_Contribution_pct
7,Fitness Smartwatch,28810.96,26.980025
10,Non-stick Cookware Set,15879.66,14.870509
5,Electric Kettle,14225.29,13.321274
2,Bluetooth Speaker,10270.51,9.617820
1,Bedsheet Set (King),8487.19,7.947830
13,Skincare Combo Kit,6054.99,5.670196
19,Yoga Mat,5897.88,5.523070
17,Wireless Earbuds Pro,4968.38,4.652640
6,Fiction Bestseller Bundle,4830.60,4.523616
8,Kids T-Shirt Pack,2911.51,2.726484


In [188]:
event_regions = (
    region_daily[
        (region_daily["Date"] >= event_start) &
        (region_daily["Date"] <= event_end)
    ]
    .groupby("Region")
    .agg(
        Event_Sales=("Sales", "sum"),
        Event_Quantity=("Quantity", "sum"),
        Event_Orders=("Orders", "sum")
    )
    .reset_index()
)

baseline_regions = (
    region_daily[
        (region_daily["Date"] >= baseline_start) &
        (region_daily["Date"] <= baseline_end)
    ]
    .groupby("Region")
    .agg(
        Baseline_Sales=("Sales", "sum"),
        Baseline_Quantity=("Quantity", "sum"),
        Baseline_Orders=("Orders", "sum")
    )
    .reset_index()
)

event_regions = event_regions.merge(
    baseline_regions,
    on="Region",
    how="left"
)

In [189]:
event_regions["Sales_Change_pct"] = (
    (event_regions["Event_Sales"] - event_regions["Baseline_Sales"])
    / event_regions["Baseline_Sales"]
) * 100

In [190]:
event_regions["Sales_Loss"] = (
    event_regions["Baseline_Sales"] - event_regions["Event_Sales"]
)

In [191]:
event_regions[
    [
        "Region",
        "Event_Sales",
        "Baseline_Sales",
        "Sales_Change_pct",
        "Sales_Loss"
    ]
].sort_values(
    "Sales_Loss",
    ascending=False
)

,Region,Event_Sales,Baseline_Sales,Sales_Change_pct,Sales_Loss
4,West,0.00,22995.16,-100.000000,22995.16
3,South,17187.69,38160.92,-54.959970,20973.23
2,North,16508.00,31383.90,-47.399781,14875.90
1,East,37147.54,44622.58,-16.751698,7475.04
0,Central,18419.68,24917.99,-26.078789,6498.31


In [200]:
def explain_event(start_date, end_date):
    
    # -------------------------
    # 1. Product-level analysis
    # -------------------------
    
    event_products = (
        product_daily[
            (product_daily["Date"] >= start_date) &
            (product_daily["Date"] <= end_date)
        ]
        .groupby("Product_Name")
        .agg(
            Event_Sales=("Sales", "sum"),
            Event_Quantity=("Quantity", "sum"),
            Event_Profit=("Profit", "sum")
        )
        .reset_index()
    )

    # Get 3 days immediately before the event
    baseline_end = pd.to_datetime(start_date) - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=2)

    baseline_products = (
        product_daily[
            (product_daily["Date"] >= baseline_start) &
            (product_daily["Date"] <= baseline_end)
        ]
        .groupby("Product_Name")
        .agg(
            Baseline_Sales=("Sales", "sum"),
            Baseline_Quantity=("Quantity", "sum"),
            Baseline_Profit=("Profit", "sum")
        )
        .reset_index()
    )

    event_products = event_products.merge(
        baseline_products,
        on="Product_Name",
        how="left"
    )

    event_products["Sales_Loss"] = (
        event_products["Baseline_Sales"] -
        event_products["Event_Sales"]
    )

    # Only products that actually lost sales
    loss_contributors = event_products[
        event_products["Sales_Loss"] > 0
    ].copy()

    total_positive_loss = loss_contributors["Sales_Loss"].sum()

    loss_contributors["Loss_Contribution_pct"] = (
        loss_contributors["Sales_Loss"] /
        total_positive_loss
    ) * 100

    loss_contributors = loss_contributors.sort_values(
        "Sales_Loss",
        ascending=False
    )

    # -------------------------
    # 2. Region-level analysis
    # -------------------------

    event_regions = (
        region_daily[
            (region_daily["Date"] >= start_date) &
            (region_daily["Date"] <= end_date)
        ]
        .groupby("Region")
        .agg(
            Event_Sales=("Sales", "sum"),
            Event_Quantity=("Quantity", "sum"),
            Event_Orders=("Orders", "sum")
        )
        .reset_index()
    )

    baseline_regions = (
        region_daily[
            (region_daily["Date"] >= baseline_start) &
            (region_daily["Date"] <= baseline_end)
        ]
        .groupby("Region")
        .agg(
            Baseline_Sales=("Sales", "sum"),
            Baseline_Quantity=("Quantity", "sum"),
            Baseline_Orders=("Orders", "sum")
        )
        .reset_index()
    )

    event_regions = event_regions.merge(
        baseline_regions,
        on="Region",
        how="left"
    )

    event_regions["Sales_Change_pct"] = (
        (event_regions["Event_Sales"] -
         event_regions["Baseline_Sales"])
        / event_regions["Baseline_Sales"]
    ) * 100

    event_regions["Sales_Loss"] = (
        event_regions["Baseline_Sales"] -
        event_regions["Event_Sales"]
    )

    event_regions = event_regions.sort_values(
        "Sales_Loss",
        ascending=False
    )

    # -------------------------
    # 3. Return the results
    # -------------------------

    return loss_contributors, event_regions

In [201]:
product_explanation, region_explanation = explain_event(
    "2024-02-09",
    "2024-02-11"
)

In [202]:
product_explanation.head(10)

,Product_Name,Event_Sales,Event_Quantity,Event_Profit,Baseline_Sales,Baseline_Quantity,Baseline_Profit,Sales_Loss,Loss_Contribution_pct
7,Fitness Smartwatch,4433.68,1,489.29,33244.64,7,4568.90,28810.96,26.980025
10,Non-stick Cookware Set,0.00,0,0.00,15879.66,5,3523.57,15879.66,14.870509
5,Electric Kettle,1314.00,1,344.44,15539.29,13,2723.59,14225.29,13.321274
2,Bluetooth Speaker,3305.43,2,453.22,13575.94,8,1571.68,10270.51,9.617820
1,Bedsheet Set (King),2893.41,2,467.40,11380.60,8,2433.42,8487.19,7.947830
13,Skincare Combo Kit,3289.44,2,1265.37,9344.43,6,3164.80,6054.99,5.670196
19,Yoga Mat,1554.98,2,454.78,7452.86,9,1917.70,5897.88,5.523070
17,Wireless Earbuds Pro,5243.28,2,242.94,10211.66,4,1374.35,4968.38,4.652640
6,Fiction Bestseller Bundle,4103.43,7,592.18,8934.03,15,1166.97,4830.60,4.523616
8,Kids T-Shirt Pack,2764.57,4,1114.12,5676.08,8,1875.24,2911.51,2.726484


In [203]:
region_explanation

,Region,Event_Sales,Event_Quantity,Event_Orders,Baseline_Sales,Baseline_Quantity,Baseline_Orders,Sales_Change_pct,Sales_Loss
4,West,0.00,0,0,22995.16,14,8,-100.000000,22995.16
3,South,17187.69,10,8,38160.92,24,14,-54.959970,20973.23
2,North,16508.00,11,6,31383.90,24,11,-47.399781,14875.90
1,East,37147.54,15,10,44622.58,24,13,-16.751698,7475.04
0,Central,18419.68,15,8,24917.99,20,6,-26.078789,6498.31


In [214]:
def explain_event(start_date, end_date):

    # Convert input dates to datetime
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # -------------------------
    # 0. Define baseline period
    # -------------------------

    baseline_end = start_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=2)

    # -------------------------
    # 1. Overall event impact
    # -------------------------

    event_overall = daily_data[
        (daily_data["Date"] >= start_date) &
        (daily_data["Date"] <= end_date)
    ]

    baseline_overall = daily_data[
        (daily_data["Date"] >= baseline_start) &
        (daily_data["Date"] <= baseline_end)
    ]

    event_sales = event_overall["Sales"].sum()
    baseline_sales = baseline_overall["Sales"].sum()

    event_profit = event_overall["Profit"].sum()
    baseline_profit = baseline_overall["Profit"].sum()

    event_quantity = event_overall["Quantity"].sum()
    baseline_quantity = baseline_overall["Quantity"].sum()

    event_orders = event_overall["Orders"].sum()
    baseline_orders = baseline_overall["Orders"].sum()

    overall_metrics = {
        "Event_Sales": event_sales,
        "Baseline_Sales": baseline_sales,
        "Sales_Change_pct": (
            (event_sales - baseline_sales) / baseline_sales
        ) * 100,

        "Event_Profit": event_profit,
        "Baseline_Profit": baseline_profit,
        "Profit_Change_pct": (
            (event_profit - baseline_profit) / baseline_profit
        ) * 100,

        "Event_Quantity": event_quantity,
        "Baseline_Quantity": baseline_quantity,
        "Quantity_Change_pct": (
            (event_quantity - baseline_quantity) / baseline_quantity
        ) * 100,

        "Event_Orders": event_orders,
        "Baseline_Orders": baseline_orders,
        "Orders_Change_pct": (
            (event_orders - baseline_orders) / baseline_orders
        ) * 100
    }

    # -------------------------
    # 2. Product-level analysis
    # -------------------------

    event_products = (
        product_daily[
            (product_daily["Date"] >= start_date) &
            (product_daily["Date"] <= end_date)
        ]
        .groupby("Product_Name")
        .agg(
            Event_Sales=("Sales", "sum"),
            Event_Quantity=("Quantity", "sum"),
            Event_Profit=("Profit", "sum")
        )
        .reset_index()
    )

    baseline_products = (
        product_daily[
            (product_daily["Date"] >= baseline_start) &
            (product_daily["Date"] <= baseline_end)
        ]
        .groupby("Product_Name")
        .agg(
            Baseline_Sales=("Sales", "sum"),
            Baseline_Quantity=("Quantity", "sum"),
            Baseline_Profit=("Profit", "sum")
        )
        .reset_index()
    )

    event_products = event_products.merge(
        baseline_products,
        on="Product_Name",
        how="left"
    )

    event_products["Sales_Loss"] = (
        event_products["Baseline_Sales"] -
        event_products["Event_Sales"]
    )

    loss_contributors = event_products[
        event_products["Sales_Loss"] > 0
    ].copy()

    total_positive_loss = loss_contributors["Sales_Loss"].sum()

    loss_contributors["Loss_Contribution_pct"] = (
        loss_contributors["Sales_Loss"] /
        total_positive_loss
    ) * 100

    loss_contributors = loss_contributors.sort_values(
        "Sales_Loss",
        ascending=False
    )

    # -------------------------
    # 3. Region-level analysis
    # -------------------------

    event_regions = (
        region_daily[
            (region_daily["Date"] >= start_date) &
            (region_daily["Date"] <= end_date)
        ]
        .groupby("Region")
        .agg(
            Event_Sales=("Sales", "sum"),
            Event_Quantity=("Quantity", "sum"),
            Event_Orders=("Orders", "sum")
        )
        .reset_index()
    )

    baseline_regions = (
        region_daily[
            (region_daily["Date"] >= baseline_start) &
            (region_daily["Date"] <= baseline_end)
        ]
        .groupby("Region")
        .agg(
            Baseline_Sales=("Sales", "sum"),
            Baseline_Quantity=("Quantity", "sum"),
            Baseline_Orders=("Orders", "sum")
        )
        .reset_index()
    )

    event_regions = event_regions.merge(
        baseline_regions,
        on="Region",
        how="left"
    )

    event_regions["Sales_Change_pct"] = (
        (event_regions["Event_Sales"] -
         event_regions["Baseline_Sales"])
        / event_regions["Baseline_Sales"]
    ) * 100

    event_regions["Sales_Loss"] = (
        event_regions["Baseline_Sales"] -
        event_regions["Event_Sales"]
    )

    event_regions = event_regions.sort_values(
        "Sales_Loss",
        ascending=False
    )

        # -------------------------
    # 4. Event classification
    # -------------------------

    declining_regions = (
        event_regions["Sales_Change_pct"] < 0
    ).sum()

    total_regions = len(event_regions)

    region_decline_pct = (
        declining_regions / total_regions
    ) * 100

    if region_decline_pct >= 80:
        regional_assessment = "Broad-based regional decline"
    elif region_decline_pct >= 50:
        regional_assessment = "Widespread regional decline"
    else:
        regional_assessment = "Region-concentrated decline"

    # -------------------------
    # 4. Return everything
    # -------------------------

    return overall_metrics, loss_contributors, event_regions, regional_assessment

In [215]:
overall_metrics, product_explanation, region_explanation = explain_event(
    "2024-02-09",
    "2024-02-11"
)

ValueError: too many values to unpack (expected 3)

In [216]:
overall_metrics

{'Event_Sales': np.float64(89262.91),
 'Baseline_Sales': np.float64(162080.55),
 'Sales_Change_pct': np.float64(-44.92682187961479),
 'Event_Profit': np.float64(22001.98),
 'Baseline_Profit': np.float64(30754.380000000005),
 'Profit_Change_pct': np.float64(-28.459035753606493),
 'Event_Quantity': np.int64(51),
 'Baseline_Quantity': np.int64(106),
 'Quantity_Change_pct': np.float64(-51.886792452830186),
 'Event_Orders': np.int64(32),
 'Baseline_Orders': np.int64(52),
 'Orders_Change_pct': np.float64(-38.46153846153847)}

In [217]:
product_explanation.head(10)

,Product_Name,Event_Sales,Event_Quantity,Event_Profit,Baseline_Sales,Baseline_Quantity,Baseline_Profit,Sales_Loss,Loss_Contribution_pct
7,Fitness Smartwatch,4433.68,1,489.29,33244.64,7,4568.90,28810.96,26.980025
10,Non-stick Cookware Set,0.00,0,0.00,15879.66,5,3523.57,15879.66,14.870509
5,Electric Kettle,1314.00,1,344.44,15539.29,13,2723.59,14225.29,13.321274
2,Bluetooth Speaker,3305.43,2,453.22,13575.94,8,1571.68,10270.51,9.617820
1,Bedsheet Set (King),2893.41,2,467.40,11380.60,8,2433.42,8487.19,7.947830
13,Skincare Combo Kit,3289.44,2,1265.37,9344.43,6,3164.80,6054.99,5.670196
19,Yoga Mat,1554.98,2,454.78,7452.86,9,1917.70,5897.88,5.523070
17,Wireless Earbuds Pro,5243.28,2,242.94,10211.66,4,1374.35,4968.38,4.652640
6,Fiction Bestseller Bundle,4103.43,7,592.18,8934.03,15,1166.97,4830.60,4.523616
8,Kids T-Shirt Pack,2764.57,4,1114.12,5676.08,8,1875.24,2911.51,2.726484


In [218]:
region_explanation

,Region,Event_Sales,Event_Quantity,Event_Orders,Baseline_Sales,Baseline_Quantity,Baseline_Orders,Sales_Change_pct,Sales_Loss
4,West,0.00,0,0,22995.16,14,8,-100.000000,22995.16
3,South,17187.69,10,8,38160.92,24,14,-54.959970,20973.23
2,North,16508.00,11,6,31383.90,24,11,-47.399781,14875.90
1,East,37147.54,15,10,44622.58,24,13,-16.751698,7475.04
0,Central,18419.68,15,8,24917.99,20,6,-26.078789,6498.31


In [220]:
overall_metrics, product_explanation, region_explanation, regional_assessment = explain_event(
    "2024-02-09",
    "2024-02-11"
)

In [221]:
print(regional_assessment)

Broad-based regional decline
